# LeetCode 465 — Optimal Account Balancing bằng Python (Hard)

> **Lưu ý: đây là bài Premium** (bài bị khóa trên LeetCode, cần tài khoản trả phí mới mở được). Phát biểu đề, ví dụ, constraints và signature dưới đây theo đúng bản LeetCode hiện hành.

Notebook này giảng nhập môn bài toán:

```python
class Solution:
    def minTransfers(self, transactions: List[List[int]]) -> int:

```

Tên bài dịch nôm na là **"Cân bằng tài khoản tối ưu"**: một nhóm bạn đã đưa tiền qua lại cho nhau nhiều lần, giờ muốn "thanh toán cho xong" bằng **ít lần chuyển khoản nhất**. Đây chính là bài toán mà các app chia tiền kiểu Splitwise phải giải mỗi ngày.

Mục tiêu của bài học:

1. Biến danh sách giao dịch lộn xộn thành **số dư ròng** (net balance) của từng người bằng `dict` / `defaultdict` — và hiểu vì sao **chỉ số dư ròng mới quan trọng**, còn lịch sử giao dịch thì có thể quên đi.
2. Tự nghĩ ra hai cách "dễ nghĩ": **thanh toán dây chuyền** (luôn dùng `k − 1` giao dịch) và **tham lam** (ghép người nợ nhiều nhất với người được nợ nhiều nhất) — rồi tự tay tìm **phản ví dụ** cho thấy chúng chưa tối ưu.
3. Nắm **insight cốt lõi**: đáp án `= k − G`, với `k` là số người có số dư khác 0 và `G` là số **nhóm con tổng bằng 0** nhiều nhất có thể tách ra. Hiểu *vì sao* một nhóm `s` người cần **đúng** `s − 1` giao dịch (lập luận "mỗi giao dịch nối hai người").
4. Học **đệ quy + quay lui** (recursion + backtracking) từ con số 0 theo nhịp *chọn → đệ quy → hoàn tác*, rồi viết **lời giải chính kinh điển**: lấy người đầu tiên còn nợ, thử cho họ thanh toán với từng người trái dấu, kèm **hai kỹ thuật cắt tỉa** (bỏ giá trị trùng, dừng sớm khi triệt tiêu đúng) — có chứng minh vì sao cắt tỉa không làm mất đáp án.
5. Học **bitmask** (dùng các bit của một số nguyên để biểu diễn một tập hợp) và giải bằng **DP bitmask** `O(2^n · n)`: nhìn lại cách dây chuyền, thử mọi thứ tự người, rồi nhận ra chỉ cần nhớ *tập* người đã xếp.
6. Đối chiếu chéo các cách trên dữ liệu ngẫu nhiên, tránh các lỗi kinh điển (quên hoàn tác, gọi đệ quy sai chỉ số, `tried` dùng chung, quên điều kiện tổng 0…) và nhận dạng pattern "chia tập nhỏ thành các nhóm" cho các bài anh em như *Partition to K Equal Sum Subsets*.

## 1. Đề bài

Đề LeetCode (dịch sát nghĩa):

> Cho mảng các giao dịch `transactions`, trong đó `transactions[i] = [from_i, to_i, amount_i]` cho biết người có `ID = from_i` **đã đưa** `amount_i` đô la cho người có `ID = to_i`.
>
> Hãy trả về **số giao dịch ít nhất** cần thực hiện để **thanh toán hết nợ** (settle the debt).

**Example 1**

```text
Input: transactions = [[0,1,10],[2,0,5]]
Output: 2
Giải thích:
Người #0 đưa người #1 $10.
Người #2 đưa người #0 $5.
Cần 2 giao dịch. Một cách thanh toán là người #1 trả cho người #0 và người #2 mỗi người $5.
```

**Example 2**

```text
Input: transactions = [[0,1,10],[1,0,1],[1,2,5],[2,0,5]]
Output: 1
Giải thích:
Người #0 đưa người #1 $10.
Người #1 đưa người #0 $1.
Người #1 đưa người #2 $5.
Người #2 đưa người #0 $5.
Vậy người #1 chỉ cần đưa người #0 $4 là mọi khoản nợ đều được thanh toán.
```

**Constraints**

- `1 <= transactions.length <= 8`
- `transactions[i].length == 3`
- `0 <= from_i, to_i < 12`
- `from_i != to_i`
- `1 <= amount_i <= 100`

### Ký hiệu dùng suốt notebook

- `P0, P1, P2, …` = người có ID `0, 1, 2, …`.
- **Số dư ròng** (net balance) của một người, ký hiệu `bal[p]`: **dương** nghĩa là người đó **đang được người khác nợ** (cần nhận lại tiền), **âm** nghĩa là người đó **đang nợ** (cần trả tiền), bằng `0` là đã sòng phẳng.
- `k` = số người có số dư **khác 0**. Vì ID `< 12` nên `k <= 12` (thực tế với 8 giao dịch thì `k` cũng không vượt quá 12).
- `debts` = danh sách các số dư khác 0, ví dụ `[5, -10, 5]`. Trong phần thuật toán ta hay gọi độ dài của nó là `n` (chính là `k`).
- "Một giao dịch" = một lần **một người chuyển một số tiền cho một người khác**, giữa **bất kỳ** hai người nào (không bắt buộc phải là hai người từng giao dịch với nhau).

### 1.1 Hiểu đề bằng hình

Vẽ Example 1. Mũi tên `A ──$x──> B` nghĩa là "A đã đưa B `x` đô" (tức là B đang nợ A `x` đô):

```text
        P0 ──$10──> P1          (P1 nợ P0 10 đô)
        P2 ──$5───> P0          (P0 nợ P2 5 đô)
```

Tính xem mỗi người "lời" hay "lỗ" bao nhiêu:

| Người | Đã đưa đi | Đã nhận về | Số dư ròng = đưa − nhận | Ý nghĩa |
|---|---:|---:|---:|---|
| P0 | 10 | 5 | **+5** | còn được nhận lại 5 |
| P1 | 0 | 10 | **−10** | còn phải trả 10 |
| P2 | 5 | 0 | **+5** | còn được nhận lại 5 |

Người P1 cầm của nhóm 10 đô, nên P1 trả cho P0 5 đô và trả cho P2 5 đô là xong: **2 giao dịch**. Có làm được bằng 1 giao dịch không? Không: một giao dịch chỉ chạm tới **2 người**, mà ở đây có **3 người** đang lệch, nên sau 1 giao dịch luôn còn ít nhất một người lệch.

Vài suy luận nhỏ mà rất quan trọng:

- Thanh toán **không** cần đi theo đúng đường cũ. Ở Example 1, ta không "hoàn" lại giao dịch `P2 → P0` mà cho P1 trả thẳng P2.
- Ở Example 2, P2 vừa đưa vừa nhận đúng 5 đô nên **số dư bằng 0** — P2 không cần tham gia gì nữa. Chỉ còn P0 (+4) và P1 (−4): 1 giao dịch.
- Đáp án có thể là `0` (ví dụ ba người đưa vòng tròn cho nhau cùng một số tiền: ai cũng đưa đi bao nhiêu nhận về bấy nhiêu).

Cell dưới khai báo các ví dụ dùng suốt bài, gồm cả một **ví dụ xuyên suốt** `tricky` mà ta sẽ dùng để "bắt lỗi" các cách giải dễ nghĩ.

In [1]:
from typing import List
from collections import defaultdict
import itertools
import random
import time

ex1 = [[0, 1, 10], [2, 0, 5]]                         # đáp án 2
ex2 = [[0, 1, 10], [1, 0, 1], [1, 2, 5], [2, 0, 5]]   # đáp án 1

# Ví dụ xuyên suốt (tự tạo, đúng constraints): đáp án 3
tricky = [[0, 2, 4], [1, 3, 2], [1, 4, 1], [2, 4, 1]]

for name, tr in [("ex1", ex1), ("ex2", ex2), ("tricky", tricky)]:
    print(f"{name:7s}: {len(tr)} giao dịch -> {tr}")

ex1    : 2 giao dịch -> [[0, 1, 10], [2, 0, 5]]
ex2    : 4 giao dịch -> [[0, 1, 10], [1, 0, 1], [1, 2, 5], [2, 0, 5]]
tricky : 4 giao dịch -> [[0, 2, 4], [1, 3, 2], [1, 4, 1], [2, 4, 1]]


## 2. Nền tảng: cộng dồn bằng `dict` và số dư ròng

### 2.1 `dict` để cộng dồn theo "chìa khóa"

`dict` (dictionary — từ điển) lưu các cặp **key → value**. Ở bài này key là **ID người**, value là **số tiền** đang cộng dồn.

Muốn cộng thêm vào một key chưa có, với `dict` thường ta phải dùng `d.get(key, 0)` (lấy giá trị, nếu chưa có thì coi là `0`). Còn `defaultdict(int)` (trong module `collections`) tự tạo giá trị `0` cho key mới — tiện hơn hẳn cho việc cộng dồn.

Ví dụ mini: cộng tiền tiêu vặt theo tên.

In [2]:
spend = [("An", 3), ("Binh", 5), ("An", 4)]

d = {}
for name, x in spend:
    d[name] = d.get(name, 0) + x     # chưa có key -> coi như 0
print("dict thường :", d)

dd = defaultdict(int)                 # key mới tự có giá trị int() = 0
for name, x in spend:
    dd[name] += x
print("defaultdict :", dict(dd))
print("dd['Chi'] chưa từng cộng ->", dd["Chi"])   # tự tạo 0, không báo lỗi KeyError

dict thường : {'An': 7, 'Binh': 5}
defaultdict : {'An': 7, 'Binh': 5}
dd['Chi'] chưa từng cộng -> 0


### 2.2 Tính số dư ròng của từng người

Quy ước dấu (giữ cố định suốt bài): giao dịch `[f, t, a]` nghĩa là `f` đưa `t` số tiền `a`, nên

- `bal[f] += a` — người `f` bỏ tiền ra, giờ **được nợ** thêm `a`;
- `bal[t] -= a` — người `t` cầm tiền, giờ **nợ** thêm `a`.

Tính tay Example 2 `[[0,1,10],[1,0,1],[1,2,5],[2,0,5]]`:

| Giao dịch | `bal[0]` | `bal[1]` | `bal[2]` |
|---|---:|---:|---:|
| bắt đầu | 0 | 0 | 0 |
| `[0,1,10]` | +10 | −10 | 0 |
| `[1,0,1]` | +9 | −9 | 0 |
| `[1,2,5]` | +9 | −4 | −5 |
| `[2,0,5]` | **+4** | **−4** | **0** |

Một tính chất luôn đúng: **tổng mọi số dư bằng 0**, vì mỗi giao dịch cộng `a` cho một người và trừ `a` của một người khác. Tiền không tự sinh ra hay mất đi — nó chỉ đổi chủ.

In [3]:
def net_balances(transactions):
    # Trả về dict {ID: số dư ròng}, sắp xếp theo ID cho dễ đọc.
    bal = defaultdict(int)
    for f, t, a in transactions:
        bal[f] += a      # f đưa tiền đi -> được nợ thêm a
        bal[t] -= a      # t cầm tiền    -> nợ thêm a
    return dict(sorted(bal.items()))


for name, tr in [("ex1", ex1), ("ex2", ex2), ("tricky", tricky)]:
    bal = net_balances(tr)
    print(f"{name:7s}: {bal}   tổng = {sum(bal.values())}")

ex1    : {0: 5, 1: -10, 2: 5}   tổng = 0
ex2    : {0: 4, 1: -4, 2: 0}   tổng = 0
tricky : {0: 4, 1: 3, 2: -3, 3: -2, 4: -2}   tổng = 0


### 2.3 Chỉ số dư ròng mới quan trọng — bỏ những người bằng 0

Câu hỏi then chốt: *hai bộ giao dịch khác nhau nhưng cho cùng số dư ròng thì đáp án có giống nhau không?* **Có.** Lý do: một kế hoạch thanh toán là hợp lệ khi và chỉ khi, sau khi thực hiện nó, **mọi người về số dư 0**. Điều kiện này chỉ phụ thuộc vào số dư ròng ban đầu, không phụ thuộc chuyện "ai đưa ai lúc nào".

Hệ quả:

1. Ta nén cả bài thành một **danh sách số nguyên có tổng bằng 0**, ví dụ Example 1 → `[5, -10, 5]`.
2. Người có số dư `0` (như P2 ở Example 2) **không bao giờ cần** tham gia giao dịch nào → bỏ khỏi danh sách. (Mục 5.1 sẽ chứng minh việc kéo thêm người số dư 0 vào chỉ làm tốn thêm giao dịch.)

Từ đây trở đi, gần như mọi thuật toán chỉ làm việc trên danh sách `debts` này.

In [4]:
def nonzero_debts(transactions):
    # Danh sách số dư KHÁC 0 (theo thứ tự ID tăng dần).
    return [b for b in net_balances(transactions).values() if b != 0]


for name, tr in [("ex1", ex1), ("ex2", ex2), ("tricky", tricky)]:
    d = nonzero_debts(tr)
    print(f"{name:7s}: debts = {d}   k = {len(d)}")

# Hai bộ giao dịch khác hẳn nhau nhưng cùng số dư ròng -> cùng một bài toán
other = [[0, 3, 4], [3, 1, 4], [2, 5, 1], [5, 2, 1]]
print("other  :", net_balances(other))
print("cùng debts với ex2?", nonzero_debts(other) == nonzero_debts(ex2), "-> đáp án cũng bằng nhau")

ex1    : debts = [5, -10, 5]   k = 3
ex2    : debts = [4, -4]   k = 2
tricky : debts = [4, 3, -3, -2, -2]   k = 5
other  : {0: 4, 1: -4, 2: 0, 3: 0, 5: 0}
cùng debts với ex2? True -> đáp án cũng bằng nhau


### 2.4 Công cụ kiểm tra: một kế hoạch thanh toán có hợp lệ không?

Một **kế hoạch** (plan) chỉ là một danh sách **giao dịch mới** cùng định dạng `[người_trả, người_nhận, số_tiền]`. Vì giao dịch mới cũng chỉ là giao dịch, kế hoạch hợp lệ khi:

```text
net_balances(transactions + plan)  có mọi giá trị bằng 0
```

Ta dùng hàm `is_settled` này suốt notebook để **tự kiểm chứng** mọi kế hoạch mà các thuật toán tạo ra, thay vì tin bằng mắt. Thử ngay với lời giải thích trong đề.

In [5]:
def is_settled(transactions, plan):
    # True nếu sau khi thực hiện thêm plan, ai cũng có số dư 0.
    return all(v == 0 for v in net_balances(transactions + plan).values())


plan1 = [[1, 0, 5], [1, 2, 5]]     # đề: P1 trả P0 và P2 mỗi người 5
plan2 = [[1, 0, 4]]                # đề: P1 trả P0 4
wrong = [[1, 0, 10]]               # P1 trả hết 10 cho P0 -> P0 dư, P2 thiếu

print("ex1 + plan1 :", is_settled(ex1, plan1), net_balances(ex1 + plan1))
print("ex2 + plan2 :", is_settled(ex2, plan2), net_balances(ex2 + plan2))
print("ex1 + wrong :", is_settled(ex1, wrong), net_balances(ex1 + wrong))

ex1 + plan1 : True {0: 0, 1: 0, 2: 0}
ex2 + plan2 : True {0: 0, 1: 0, 2: 0}
ex1 + wrong : False {0: -5, 1: 0, 2: 5}


## 3. Cách 1 — Thanh toán dây chuyền (luôn đúng, nhưng chưa chắc ít nhất)

### Ý tưởng tự nhiên nhất

Xếp những người còn lệch thành một hàng `p1, p2, …, pk`. Người đầu hàng **chuyển toàn bộ phần lệch của mình cho người kế tiếp**: sau giao dịch đó người đầu về 0, còn người kế tiếp "gánh" thêm phần lệch. Lặp lại đến cuối hàng. Người cuối cùng tự động về 0 (vì tổng luôn bằng 0).

Biến `carry` = tổng số dư của các người đã đi qua (gồm cả người hiện tại) = phần lệch đang được "chuyền tay":

- `carry > 0`: người hiện tại đang được nợ `carry` → người kế tiếp **trả** cho họ `carry`;
- `carry < 0`: người hiện tại đang nợ → họ **trả** người kế tiếp `-carry`;
- `carry == 0`: người hiện tại đã sòng phẳng → **không cần giao dịch**, tiết kiệm được 1 lần!

Tính tay với `tricky`, debts `[+4, +3, −3, −2, −2]` theo thứ tự P0, P1, P2, P3, P4:

| Bước | Người hiện tại | `carry` sau khi cộng | Giao dịch | 
|---|---|---:|---|
| 1 | P0 (+4) | +4 | P1 trả P0 4 |
| 2 | P1 (+3) | +7 | P2 trả P1 7 |
| 3 | P2 (−3) | +4 | P3 trả P2 4 |
| 4 | P3 (−2) | +2 | P4 trả P3 2 |
| — | P4 (−2) | 0 | (người cuối, tự xong) |

→ **4 giao dịch** `= k − 1`. Đúng, nhưng có phải ít nhất không? Hãy đặt câu hỏi đó ở cuối mục 3.

In [6]:
def settle_chain(balances, order=None):
    # Thanh toán dây chuyền theo thứ tự `order` (mặc định: các người số dư khác 0, ID tăng dần).
    if order is None:
        order = [p for p, b in balances.items() if b != 0]
    plan = []
    carry = 0
    for idx in range(len(order) - 1):
        p, nxt = order[idx], order[idx + 1]
        carry += balances[p]
        if carry > 0:
            plan.append([nxt, p, carry])     # nxt trả p
        elif carry < 0:
            plan.append([p, nxt, -carry])    # p trả nxt
        # carry == 0: p đã sòng phẳng, không cần giao dịch
    return plan


for name, tr in [("ex1", ex1), ("ex2", ex2), ("tricky", tricky)]:
    plan = settle_chain(net_balances(tr))
    print(f"{name:7s}: {len(plan)} giao dịch {plan}  hợp lệ? {is_settled(tr, plan)}")

ex1    : 2 giao dịch [[1, 0, 5], [1, 2, 5]]  hợp lệ? True
ex2    : 1 giao dịch [[1, 0, 4]]  hợp lệ? True
tricky : 4 giao dịch [[1, 0, 4], [2, 1, 7], [3, 2, 4], [4, 3, 2]]  hợp lệ? True


### 3.1 Thứ tự xếp hàng quyết định số giao dịch

Ở mỗi bước, giao dịch chỉ bị bỏ qua khi `carry == 0`, mà `carry` chính là **tổng tiền tố** (prefix sum — tổng của các phần tử đầu hàng tính tới vị trí hiện tại). Vậy:

```text
số giao dịch dây chuyền = k − (số vị trí có tổng tiền tố bằng 0)
```

(vị trí cuối luôn có tổng tiền tố bằng 0, nên tối đa là `k − 1`.)

Với `tricky`, đổi thứ tự thành P1, P2, P0, P3, P4 tức `[+3, −3, +4, −2, −2]`: tổng tiền tố là `3, 0, 4, 2, 0` → có **hai** số 0 → chỉ còn `5 − 2 = 3` giao dịch. Cell dưới kiểm chứng và in tổng tiền tố bằng `itertools.accumulate` (hàm cộng dồn có sẵn của Python).

In [7]:
bal_t = net_balances(tricky)
for order in ([0, 1, 2, 3, 4], [1, 2, 0, 3, 4]):
    prefix = list(itertools.accumulate(bal_t[p] for p in order))
    plan = settle_chain(bal_t, order)
    print(f"thứ tự {order}: tổng tiền tố {prefix} -> {len(plan)} giao dịch {plan}  hợp lệ? {is_settled(tricky, plan)}")

thứ tự [0, 1, 2, 3, 4]: tổng tiền tố [4, 7, 4, 2, 0] -> 4 giao dịch [[1, 0, 4], [2, 1, 7], [3, 2, 4], [4, 3, 2]]  hợp lệ? True
thứ tự [1, 2, 0, 3, 4]: tổng tiền tố [3, 0, 4, 2, 0] -> 3 giao dịch [[2, 1, 3], [3, 0, 4], [4, 3, 2]]  hợp lệ? True


**Cách 1 vướng ở đâu?** Cách dây chuyền **luôn đúng** và chạy `O(k)`, nhưng số giao dịch phụ thuộc vào thứ tự: thứ tự "ngây thơ" cho 4, thứ tự khéo cho 3. Muốn tối ưu, ta phải biết **xếp hàng thế nào** — đây là manh mối ta sẽ quay lại ở Cách 4 (DP bitmask).

Trước đó, hãy thử một ý tưởng mà gần như ai cũng nghĩ tới tiếp theo: **tham lam**.

## 4. Cách 2 — Tham lam: ghép người nợ nhiều nhất với người được nợ nhiều nhất

### Ý tưởng

Lặp lại cho tới khi mọi người về 0:

1. Chọn `cred` = người có số dư **dương lớn nhất** (được nợ nhiều nhất), `debt` = người có số dư **âm nhỏ nhất** (nợ nhiều nhất).
2. Cho `debt` trả `cred` một khoản `x = min(bal[cred], -bal[debt])` — tức là trả tới khi một trong hai người về 0.
3. Xóa người đã về 0.

Mỗi vòng có ít nhất một người về 0, nên tối đa `k − 1` giao dịch — không bao giờ tệ hơn dây chuyền. Nghe rất hợp lý: "giải quyết món nợ to nhất trước". Hàm dưới có tham số `verbose` để in từng bước (dùng hàm `max(bal, key=bal.get)` = lấy **key** có **value** lớn nhất).

In [8]:
def settle_greedy(balances, verbose=False):
    bal = {p: b for p, b in balances.items() if b != 0}
    plan = []
    while bal:
        cred = max(bal, key=bal.get)        # dương lớn nhất
        debt = min(bal, key=bal.get)        # âm nhỏ nhất
        x = min(bal[cred], -bal[debt])
        plan.append([debt, cred, x])        # debt trả cred x đô
        bal[cred] -= x
        bal[debt] += x
        if verbose:
            print(f"  P{debt} trả P{cred} {x:>2} -> còn lại {bal}")
        bal = {p: b for p, b in bal.items() if b != 0}
    return plan


print("tricky, bắt đầu", net_balances(tricky))
plan = settle_greedy(net_balances(tricky), verbose=True)
print("=> tham lam:", len(plan), "giao dịch, hợp lệ?", is_settled(tricky, plan))

# Một trường hợp tham lam THẮNG dây chuyền theo thứ tự ID
bal_ab = {0: 5, 1: 3, 2: -5, 3: -3}
print("\n{0:5,1:3,2:-5,3:-3}: dây chuyền", len(settle_chain(bal_ab)), "| tham lam", len(settle_greedy(bal_ab)))

tricky, bắt đầu {0: 4, 1: 3, 2: -3, 3: -2, 4: -2}
  P2 trả P0  3 -> còn lại {0: 1, 1: 3, 2: 0, 3: -2, 4: -2}
  P3 trả P1  2 -> còn lại {0: 1, 1: 1, 3: 0, 4: -2}
  P4 trả P0  1 -> còn lại {0: 0, 1: 1, 4: -1}
  P4 trả P1  1 -> còn lại {1: 0, 4: 0}
=> tham lam: 4 giao dịch, hợp lệ? True

{0:5,1:3,2:-5,3:-3}: dây chuyền 3 | tham lam 2


### 4.1 Phản ví dụ: tham lam vẫn chưa tối ưu

Với `tricky` = `[+4, +3, −3, −2, −2]`, tham lam dùng **4** giao dịch. Nhưng nhìn kỹ, ta có cách chỉ **3** giao dịch:

- P2 trả P1 3 đô → cặp `(+3, −3)` triệt tiêu nhau ngay, **1 giao dịch** cho 2 người;
- P3 trả P0 2 đô, P4 trả P0 2 đô → nhóm `(+4, −2, −2)` xong bằng **2 giao dịch** cho 3 người.

Tham lam hỏng ở bước đầu tiên: nó ghép "to nhất với to nhất" là P0 (+4) với P2 (−3), làm **vỡ** cặp `(+3, −3)` vốn có thể tự giải quyết với nhau bằng đúng một giao dịch. Sau đó cả 5 người dính vào nhau như một khối, và một khối 5 người thì cần 4 giao dịch.

> **Mấu chốt cải tiến**: điều làm giảm số giao dịch không phải "món nợ to hay nhỏ", mà là **tách được bao nhiêu nhóm người tự cân bằng với nhau** (nhóm có tổng số dư bằng 0). Mục 5 biến nhận xét này thành công thức chính xác.

| Cách | `tricky` | Đúng (hợp lệ)? | Tối ưu? |
|---|:---:|:---:|:---:|
| Dây chuyền thứ tự ID | 4 | ✅ | ❌ |
| Tham lam lớn nhất | 4 | ✅ | ❌ |
| Tối ưu | **3** | ✅ | ✅ |

## 5. Insight cốt lõi: nhóm con tổng bằng 0

**Định nghĩa.** Một **nhóm tổng 0** là một tập người (khác rỗng) có tổng số dư bằng 0. Ví dụ với `tricky`: `{P1, P2}` (3 − 3 = 0), `{P0, P3, P4}` (4 − 2 − 2 = 0), và cả `{P0, P1, P2, P3, P4}`.

**Định lý (chìa khóa của cả bài).** Gọi `k` = số người có số dư khác 0, và `G` = số nhóm **nhiều nhất** khi ta chia `k` người này thành các nhóm tổng 0 rời nhau (mỗi người thuộc đúng một nhóm). Khi đó

```text
số giao dịch ít nhất  =  k − G
```

Kiểm tra trên các ví dụ:

| Ví dụ | debts | Cách chia tốt nhất | `k` | `G` | `k − G` |
|---|---|---|---:|---:|---:|
| ex1 | `[5, −10, 5]` | `{5, −10, 5}` (không tách được) | 3 | 1 | **2** |
| ex2 | `[4, −4]` | `{4, −4}` | 2 | 1 | **1** |
| tricky | `[4, 3, −3, −2, −2]` | `{3, −3}`, `{4, −2, −2}` | 5 | 2 | **3** |

Định lý đến từ hai mệnh đề về **một nhóm tổng 0 gồm `s` người**: (a) giải quyết được bằng `s − 1` giao dịch; (b) không thể ít hơn `s − 1`. Ta chứng minh ở mục 5.1.

### 5.1 Vì sao một nhóm `s` người cần **đúng** `s − 1` giao dịch?

**(a) Đủ `s − 1`:** dùng chính Cách 1 (dây chuyền) bên trong nhóm: mỗi người trừ người cuối tốn đúng 1 giao dịch → `s − 1`. Làm vậy cho từng nhóm: tổng cộng `Σ (s_j − 1) = k − G`.

**(b) Không thể ít hơn:** hình dung mỗi giao dịch là một **sợi dây nối hai người**. Ban đầu mỗi người đứng riêng thành một "cụm". Mỗi sợi dây nối **tối đa hai cụm** thành một, nên sau `T` giao dịch, `k` người còn **ít nhất `k − T` cụm**.

Tiền chỉ chạy dọc theo các sợi dây, nên trong mỗi cụm, tổng số dư ban đầu phải bằng 0 (sau khi thanh toán ai cũng về 0, mà giao dịch trong cụm không làm đổi tổng của cụm). Vậy các cụm chính là một cách chia thành **nhóm tổng 0** → số cụm `<= G`. Ghép lại: `k − T <= số cụm <= G`, tức `T >= k − G`.

(Nếu kế hoạch kéo thêm người số dư 0 vào, họ chỉ làm tăng số người cần nối mà không tạo thêm cụm có ích — lập luận trên vẫn đúng và còn tốn hơn. Đó là lý do bỏ người số dư 0 ở mục 2.3.)

Ví dụ số: nhóm `{P0(+4), P3(−2), P4(−2)}` có `s = 3`. Nếu chỉ dùng 1 giao dịch, nó nối được 2 người, người thứ ba vẫn đứng riêng với số dư khác 0 → chưa xong. Phải có 2 giao dịch.

Cell dưới **đếm cụm** sau từng giao dịch của một kế hoạch, dùng một kỹ thuật đơn giản: mỗi người mang một "nhãn cụm"; nối hai người thì đổi toàn bộ nhãn của cụm này sang nhãn của cụm kia.

In [9]:
def count_clusters(people, plan, verbose=False):
    label = {p: p for p in people}           # ban đầu mỗi người là một cụm riêng
    for a, b, x in plan:
        old, new = label[b], label[a]
        for p in label:                        # gộp cụm chứa b vào cụm chứa a
            if label[p] == old:
                label[p] = new
        if verbose:
            print(f"  giao dịch P{a}->P{b} ({x}): số cụm = {len(set(label.values()))}")
    clusters = defaultdict(list)
    for p, lab in label.items():
        clusters[lab].append(p)
    return list(clusters.values())


bal_t = net_balances(tricky)
best_plan = [[2, 1, 3], [3, 0, 2], [4, 0, 2]]      # kế hoạch 3 giao dịch ở mục 4.1
print("hợp lệ?", is_settled(tricky, best_plan))
groups = count_clusters(list(bal_t), best_plan, verbose=True)
for g in groups:
    print("  cụm", g, "tổng số dư =", sum(bal_t[p] for p in g))
print("k =", len(bal_t), "| số cụm =", len(groups), "| số giao dịch =", len(best_plan), "= k − số cụm")

hợp lệ? True
  giao dịch P2->P1 (3): số cụm = 4
  giao dịch P3->P0 (2): số cụm = 3
  giao dịch P4->P0 (2): số cụm = 2
  cụm [0, 3, 4] tổng số dư = 0
  cụm [1, 2] tổng số dư = 0
k = 5 | số cụm = 2 | số giao dịch = 3 = k − số cụm


### 5.2 Bài toán mới: chia thành **nhiều nhóm tổng 0 nhất**

Nhờ định lý, bài toán "ít giao dịch nhất" trở thành:

> Cho danh sách `debts` (tổng bằng 0, không có số 0), chia nó thành **nhiều nhóm nhất** sao cho mỗi nhóm có tổng bằng 0. Đáp án `= len(debts) − (số nhóm)`.

Cách "vét cạn" thẳng thắn nhất cho bài toán mới: lấy phần tử đầu tiên, **thử mọi tập con** của các phần tử còn lại để ghép cùng nhóm với nó; nếu tổng nhóm bằng 0 thì đệ quy trên phần còn lại. Hàm `itertools.combinations(range(m), r)` liệt kê mọi cách chọn `r` chỉ số trong `m` chỉ số.

Cách này chậm (gần `3^n` bước) nhưng **rất dễ tin là đúng**, nên ta giữ nó làm "trọng tài" để đối chiếu các cách nhanh hơn ở mục 11.

In [10]:
def max_zero_groups_bruteforce(debts):
    if not debts:
        return 0
    first, rest = debts[0], debts[1:]
    best = 0
    for r in range(len(rest) + 1):
        for chosen in itertools.combinations(range(len(rest)), r):
            if first + sum(rest[j] for j in chosen) == 0:          # nhóm chứa `first` có tổng 0
                left = [rest[j] for j in range(len(rest)) if j not in chosen]
                best = max(best, 1 + max_zero_groups_bruteforce(left))
    return best


def min_transfers_partition_bruteforce(debts):
    return len(debts) - max_zero_groups_bruteforce(debts)


for name, tr in [("ex1", ex1), ("ex2", ex2), ("tricky", tricky)]:
    d = nonzero_debts(tr)
    print(f"{name:7s}: debts={d}  G={max_zero_groups_bruteforce(d)}  đáp án={min_transfers_partition_bruteforce(d)}")

ex1    : debts=[5, -10, 5]  G=1  đáp án=2
ex2    : debts=[4, -4]  G=1  đáp án=1
tricky : debts=[4, 3, -3, -2, -2]  G=2  đáp án=3


## 6. Nền tảng: đệ quy và quay lui (backtracking)

**Đệ quy** (recursion) là khi một hàm **gọi lại chính nó** trên một bài toán nhỏ hơn, và có một **điểm dừng** (base case) cho bài toán nhỏ nhất.

**Quay lui** (backtracking) là cách dùng đệ quy để **thử lần lượt mọi lựa chọn**, theo một nhịp ba bước luôn lặp lại:

```text
for mỗi lựa chọn hợp lệ:
    1. CHỌN     : sửa trạng thái chung (thêm vào path, cộng tiền cho ai đó…)
    2. ĐỆ QUY   : giải phần còn lại với trạng thái đã sửa
    3. HOÀN TÁC : trả trạng thái về y như trước bước 1, để thử lựa chọn kế tiếp
```

Bước 3 là linh hồn của quay lui: vì mọi nhánh **dùng chung một biến trạng thái** (một list), nhánh trước phải "dọn dẹp" thì nhánh sau mới bắt đầu từ trạng thái sạch.

Ví dụ mini: liệt kê mọi tập con của `[3, -3, 1]` và đánh dấu tập con có tổng 0. Ở mỗi phần tử ta có 2 lựa chọn: **không lấy** hoặc **lấy** (lấy = `path.append`, hoàn tác = `path.pop`).

Nửa sau của cell chạy thêm một bản **quên hoàn tác** (`path.pop()`) để thấy tận mắt: phần tử đã chọn ở nhánh trước "rò rỉ" sang nhánh sau, các "tập con" dài ra và lặp lại — sai bét. Đây cũng chính là Lỗi 2 ở mục 14, khi ta quên trả lại số nợ đã chuyển.

In [11]:
def subsets_backtrack(nums, path=None, i=0, out=None):
    if path is None:
        path, out = [], []
    if i == len(nums):                    # base case: đã quyết định xong mọi phần tử
        out.append(list(path))            # chép lại! (path sẽ còn bị sửa tiếp)
        return out
    subsets_backtrack(nums, path, i + 1, out)       # lựa chọn 1: KHÔNG lấy nums[i]
    path.append(nums[i])                            # lựa chọn 2: CHỌN nums[i]
    subsets_backtrack(nums, path, i + 1, out)       #   ĐỆ QUY
    path.pop()                                      #   HOÀN TÁC
    return out


for s in subsets_backtrack([3, -3, 1]):
    print(f"{str(s):12s} tổng = {sum(s):>2}", "<- tổng 0" if s and sum(s) == 0 else "")


# Thí nghiệm: QUÊN hoàn tác (thiếu path.pop())
def subsets_forget_pop(nums, path, i, out):
    if i == len(nums):
        out.append(list(path))
        return
    subsets_forget_pop(nums, path, i + 1, out)
    path.append(nums[i])
    subsets_forget_pop(nums, path, i + 1, out)
    # quên path.pop()  <- lỗi!


out = []
subsets_forget_pop([3, -3, 1], [], 0, out)
print("\nBản quên pop:", out)
print("Có 8 'tập con' nhưng nhiều tập dài hơn 3 phần tử và lặp lại -> sai.")

[]           tổng =  0 
[1]          tổng =  1 
[-3]         tổng = -3 
[-3, 1]      tổng = -2 
[3]          tổng =  3 
[3, 1]       tổng =  4 
[3, -3]      tổng =  0 <- tổng 0
[3, -3, 1]   tổng =  1 

Bản quên pop: [[], [1], [1, -3], [1, -3, 1], [1, -3, 1, 3], [1, -3, 1, 3, 1], [1, -3, 1, 3, 1, -3], [1, -3, 1, 3, 1, -3, 1]]
Có 8 'tập con' nhưng nhiều tập dài hơn 3 phần tử và lặp lại -> sai.


## 7. Cách 3 (cách chính) — Quay lui: "người đầu tiên còn lệch thanh toán với một người trái dấu"

### 7.1 Ý tưởng

Làm việc trên danh sách `debts` (chỉ số `0..n-1`). Hàm `dfs(start)` trả về **số giao dịch ít nhất** để đưa mọi người từ vị trí `start` trở đi về 0.

1. Bỏ qua những người đã về 0: `while start < n and debts[start] == 0: start += 1`.
2. Nếu `start == n`: ai cũng xong → trả `0` (base case).
3. Ngược lại, người `start` **chắc chắn phải** tham gia ít nhất một giao dịch. Ta thử cho người `start` **thanh toán toàn bộ** phần lệch của mình với **một** người `i > start` **trái dấu** (`debts[i] * debts[start] < 0`):
   - giao dịch đó làm `start` về 0 và người `i` gánh phần lệch: `debts[i] += debts[start]`;
   - tốn `1` giao dịch + `dfs(start + 1)` cho phần còn lại;
   - **hoàn tác**: `debts[i] -= debts[start]`, rồi thử người `i` kế tiếp.
4. Lấy `min` trên mọi lựa chọn `i`.

Chiều tiền của "một giao dịch giữa `start` và `i`":

| `debts[start]` | Nghĩa | Giao dịch | `debts[i]` sau đó |
|---|---|---|---|
| `+4` (được nợ 4) | cần nhận 4 | người `i` trả người `start` 4 | `debts[i] + 4` |
| `−3` (nợ 3) | cần trả 3 | người `start` trả người `i` 3 | `debts[i] − 3` |

**Bất biến (invariant)** giúp code không bao giờ "kẹt": khi gọi `dfs(start)`, tổng `debts[start:]` luôn bằng 0 (vì phần lệch của những người trước đã được chuyển hết vào những người sau). Nên nếu `debts[start] != 0` thì **luôn tồn tại** người trái dấu phía sau.

### Vì sao chỉ cần thử kiểu "chuyển toàn bộ cho một người trái dấu"?

Xét cách chia tối ưu thành `G` nhóm tổng 0, và nhóm `X` chứa người `start` (có `s` người). Trong `X` chắc chắn có người `i` trái dấu với `start` (tổng nhóm bằng 0). Cho `start` chuyển toàn bộ cho `i`: tốn 1 giao dịch, nhóm `X` còn `s − 1` người lệch và **vẫn có tổng 0**. Lặp lại, nhóm `X` tốn đúng `s − 1` giao dịch — khớp với định lý. Vì `dfs` **thử mọi** `i` trái dấu, nó chắc chắn đi qua nhánh "đúng" này, nên tìm được `k − G`. Ngược lại mọi nhánh đều là kế hoạch hợp lệ, nên không thể cho số nhỏ hơn tối ưu.

### 7.2 Xem một bước "chọn" bằng tay

Với `tricky` = `[+4, +3, −3, −2, −2]`, người `start = 0` có `+4`. Những người trái dấu phía sau là vị trí 2, 3, 4. Mỗi lựa chọn tạo ra một trạng thái mới cho phần còn lại `debts[1:]`:

| Chọn `i` | Giao dịch | `debts[1:]` sau khi chọn | Tổng |
|---|---|---|---:|
| 2 (−3) | P2 trả P0 4 | `[3, 1, −2, −2]` | 0 |
| 3 (−2) | P3 trả P0 4 | `[3, −3, 2, −2]` | 0 |
| 4 (−2) | P4 trả P0 4 | `[3, −3, −2, 2]` | 0 |

Để ý: lựa chọn 3 và 4 tạo ra hai danh sách chỉ khác **thứ tự** — cùng một bài toán! Đây là manh mối cho kỹ thuật cắt tỉa thứ nhất (mục 7.4).

In [12]:
debts = nonzero_debts(tricky)
start = 0
for i in range(start + 1, len(debts)):
    if debts[i] * debts[start] < 0:
        debts[i] += debts[start]                  # CHỌN
        print(f"chọn i={i}: debts[1:] = {debts[1:]}  tổng = {sum(debts[1:])}")
        debts[i] -= debts[start]                  # HOÀN TÁC
print("sau khi hoàn tác hết:", debts)

chọn i=2: debts[1:] = [3, 1, -2, -2]  tổng = 0
chọn i=3: debts[1:] = [3, -3, 2, -2]  tổng = 0
chọn i=4: debts[1:] = [3, -3, -2, 2]  tổng = 0
sau khi hoàn tác hết: [4, 3, -3, -2, -2]


### 7.3 Code quay lui thuần (chưa cắt tỉa)

Ta thêm một bộ đếm `calls` (số lần gọi `dfs`) để lát nữa so sánh với bản cắt tỉa. `float("inf")` là "dương vô cùng", dùng làm giá trị khởi đầu cho `min`.

In [13]:
def min_transfers_backtrack_plain(debts, stats=None):
    debts = list(debts)          # chép ra để không sửa list của người gọi
    n = len(debts)

    def dfs(start):
        if stats is not None:
            stats["calls"] += 1
        while start < n and debts[start] == 0:
            start += 1
        if start == n:
            return 0
        best = float("inf")
        for i in range(start + 1, n):
            if debts[i] * debts[start] < 0:          # chỉ ghép với người trái dấu
                debts[i] += debts[start]              # CHỌN: start chuyển hết cho i
                best = min(best, 1 + dfs(start + 1))  # ĐỆ QUY
                debts[i] -= debts[start]              # HOÀN TÁC
        return best

    return dfs(0)


for name, tr in [("ex1", ex1), ("ex2", ex2), ("tricky", tricky)]:
    st = {"calls": 0}
    print(f"{name:7s}: {min_transfers_backtrack_plain(nonzero_debts(tr), st)} giao dịch, số lần gọi dfs = {st['calls']}")

ex1    : 2 giao dịch, số lần gọi dfs = 3
ex2    : 1 giao dịch, số lần gọi dfs = 2
tricky : 3 giao dịch, số lần gọi dfs = 24


### 7.4 Quay lui thuần vướng ở đâu? → Hai kỹ thuật cắt tỉa

Xét `debts = [5, 5, 5, −3, −3, −3, −3, −3]` (8 người, tổng 0). Người đầu `+5` có **năm** người trái dấu, nhưng cả năm đều là `−3`: chọn ai thì phần còn lại cũng là **cùng một multiset** (tập có lặp) giá trị, chỉ khác thứ tự. Quay lui thuần giải lại cùng một bài toán con 5 lần, rồi ở tầng dưới lại lặp tiếp… số nhánh nhân lên rất nhanh.

**Cắt tỉa 1 — bỏ giá trị trùng ở cùng một tầng.** Trong vòng `for` của một lần gọi `dfs`, giữ một `set` tên `tried` các **giá trị** `debts[i]` đã thử. Gặp lại giá trị đã có thì bỏ qua: hai lựa chọn có cùng `debts[i]` cho ra cùng multiset → cùng đáp án. (Lưu ý: `tried` phải tạo **mới trong mỗi lần gọi**, xem Lỗi 4.)

**Cắt tỉa 2 — dừng sớm khi triệt tiêu đúng.** Nếu `debts[start] + debts[i] == 0` (một cặp tự cân bằng, ví dụ `+3` và `−3`), thì nhánh này đã là **tốt nhất có thể** ở tầng hiện tại → `break`, không cần thử các `i` sau.

*Vì sao cắt tỉa 2 không làm mất đáp án?* Gọi `a = start`, `b = i` với `debts[a] + debts[b] = 0`. Lấy một cách chia tối ưu bất kỳ:
- Nếu `a`, `b` cùng nhóm `A`: tách `A` thành `{a, b}` và `A \ {a, b}` (phần còn lại vẫn tổng 0, có thể rỗng) → số nhóm **không giảm**.
- Nếu `a ∈ A`, `b ∈ B` khác nhóm: thay `A, B` bằng `{a, b}` và `(A ∪ B) \ {a, b}` (tổng `0 + 0 − 0 = 0`) → số nhóm **giữ nguyên**.

Vậy luôn có một cách chia tối ưu chứa nhóm `{a, b}`. Trong cách chia đó, `a` chuyển hết cho `b` là đúng bước đầu của nhóm `{a, b}` → nhánh `i = b` đạt tối ưu, các nhánh khác không thể tốt hơn.

Ví dụ số: với `tricky`, khi `start` là P1 (`+3`) và còn `[3, −3, 2, −2]`, ghép với `−3` cho cặp triệt tiêu → `break` ngay, không cần thử `−2`.

In [14]:
def min_transfers_backtrack(debts, stats=None):
    debts = list(debts)
    n = len(debts)

    def dfs(start):
        if stats is not None:
            stats["calls"] += 1
        while start < n and debts[start] == 0:
            start += 1
        if start == n:
            return 0
        best = float("inf")
        tried = set()                                   # giá trị đã thử ở TẦNG NÀY
        for i in range(start + 1, n):
            if debts[i] * debts[start] < 0 and debts[i] not in tried:
                tried.add(debts[i])
                debts[i] += debts[start]                # CHỌN
                best = min(best, 1 + dfs(start + 1))    # ĐỆ QUY
                debts[i] -= debts[start]                # HOÀN TÁC
                if debts[i] + debts[start] == 0:        # cặp triệt tiêu đúng -> tối ưu rồi
                    break
        return best

    return dfs(0)


tests = [nonzero_debts(tricky), [3, 3, 3, 3, -4, -4, -4], [5, 5, 5, -3, -3, -3, -3, -3], [2, 2, 2, 2, -1, -1, -1, -1, -4]]
print(f"{'debts':38s} {'đáp án':>6s} {'gọi (thuần)':>12s} {'gọi (cắt tỉa)':>14s}")
for d in tests:
    s1, s2 = {"calls": 0}, {"calls": 0}
    r1 = min_transfers_backtrack_plain(d, s1)
    r2 = min_transfers_backtrack(d, s2)
    assert r1 == r2
    print(f"{str(d):38s} {r2:>6d} {s1['calls']:>12d} {s2['calls']:>14d}")

debts                                  đáp án  gọi (thuần)  gọi (cắt tỉa)
[4, 3, -3, -2, -2]                          3           24              8
[3, 3, 3, 3, -4, -4, -4]                    6          235             22
[5, 5, 5, -3, -3, -3, -3, -3]               7         1286             11
[2, 2, 2, 2, -1, -1, -1, -1, -4]            6         1467             53


## 8. Trace từng bước Cách 3 trên `tricky`

Hàm `trace_backtrack` in mỗi lần gọi `dfs` (thụt lề theo độ sâu), mỗi lựa chọn được thử, kết quả nhánh và các lần cắt tỉa. Để dễ đọc, nó chỉ in phần **còn hoạt động** `debts[start:]`.

Dự đoán trước khi chạy (đối chiếu với output bên dưới):

| Tầng | Người `start` | Phần còn lại | Lựa chọn | Kết quả |
|---|---|---|---|---|
| 0 | P0 `+4` | `[4, 3, −3, −2, −2]` | P2 (−3) | `1 + 3 = 4` |
| 0 | P0 `+4` | | P3 (−2) | `1 + 2 = 3` → best = 3 |
| 0 | P0 `+4` | | P4 (−2) | **bỏ**: giá trị −2 đã thử |
| 1 (nhánh P3) | P1 `+3` | `[3, −3, 2, −2]` | P2 (−3) | triệt tiêu đúng → `1 + 1 = 2`, **break** |
| 2 | P3 `+2` | `[2, −2]` | P4 (−2) | triệt tiêu đúng → `1 + 0 = 1`, break |

Đáp án `3`: P3 trả P0 4 (P3 thành `+2`), P2 trả P1 3, P4 trả P3 2.

In [15]:
def trace_backtrack(debts):
    debts = list(debts)
    n = len(debts)

    def dfs(start, depth):
        pad = "    " * depth
        while start < n and debts[start] == 0:
            start += 1
        if start == n:
            print(f"{pad}mọi người đã về 0 -> 0")
            return 0
        print(f"{pad}dfs: P{start} còn {debts[start]:+d}, phần còn lại {debts[start:]}")
        best = float("inf")
        tried = set()
        for i in range(start + 1, n):
            if debts[i] * debts[start] >= 0:
                continue
            if debts[i] in tried:
                print(f"{pad}  bỏ P{i} ({debts[i]:+d}): giá trị đã thử")
                continue
            tried.add(debts[i])
            print(f"{pad}  thử P{start}<->P{i}: P{i} {debts[i]:+d} -> {debts[i] + debts[start]:+d}")
            debts[i] += debts[start]
            sub = dfs(start + 1, depth + 1)
            debts[i] -= debts[start]
            best = min(best, 1 + sub)
            print(f"{pad}  => nhánh P{i}: 1 + {sub} = {1 + sub}, best = {best}")
            if debts[i] + debts[start] == 0:
                print(f"{pad}  P{start} và P{i} triệt tiêu đúng -> break")
                break
        return best

    ans = dfs(0, 0)
    print("ĐÁP ÁN:", ans)
    return ans


trace_backtrack(nonzero_debts(tricky))

plan_bt = [[3, 0, 4], [2, 1, 3], [4, 3, 2]]      # kế hoạch truy ngược từ nhánh tốt nhất (xem 8.1)
print("kế hoạch", plan_bt, "hợp lệ?", is_settled(tricky, plan_bt))

dfs: P0 còn +4, phần còn lại [4, 3, -3, -2, -2]
  thử P0<->P2: P2 -3 -> +1
    dfs: P1 còn +3, phần còn lại [3, 1, -2, -2]
      thử P1<->P3: P3 -2 -> +1
        dfs: P2 còn +1, phần còn lại [1, 1, -2]
          thử P2<->P4: P4 -2 -> -1
            dfs: P3 còn +1, phần còn lại [1, -1]
              thử P3<->P4: P4 -1 -> +0
                mọi người đã về 0 -> 0
              => nhánh P4: 1 + 0 = 1, best = 1
              P3 và P4 triệt tiêu đúng -> break
          => nhánh P4: 1 + 1 = 2, best = 2
      => nhánh P3: 1 + 2 = 3, best = 3
      bỏ P4 (-2): giá trị đã thử
  => nhánh P2: 1 + 3 = 4, best = 4
  thử P0<->P3: P3 -2 -> +2
    dfs: P1 còn +3, phần còn lại [3, -3, 2, -2]
      thử P1<->P2: P2 -3 -> +0
        dfs: P3 còn +2, phần còn lại [2, -2]
          thử P3<->P4: P4 -2 -> +0
            mọi người đã về 0 -> 0
          => nhánh P4: 1 + 0 = 1, best = 1
          P3 và P4 triệt tiêu đúng -> break
      => nhánh P2: 1 + 1 = 2, best = 2
      P1 và P2 triệt tiêu đúng -> break
  =>

### 8.1 Đọc trace

- **Nhánh P2 cho 4** vì ghép P0 (+4) với P2 (−3) làm P2 thành `+1` — đúng kiểu "phá cặp" của tham lam; phần còn lại `[3, 1, −2, −2]` không còn tách được nhóm tổng 0 nhỏ nào, nên tốn `k − 1 = 3` giao dịch nữa.
- **Nhánh P3 cho 3**: P3 nhận phần `+4` của P0 thành `+2`; ngay sau đó P1 (+3) gặp P2 (−3) là cặp triệt tiêu → `break` (cắt tỉa 2), và P3 (+2) gặp P4 (−2) cũng triệt tiêu.
- **P4 bị bỏ** ở tầng 0 nhờ cắt tỉa 1: P4 có cùng giá trị `−2` với P3 đã thử.
- Mỗi dòng `=> nhánh` cho thấy giá trị `best` chỉ **giảm dần**, và đáp án cuối là `min` của các nhánh.

- Truy ngược kế hoạch của nhánh tốt nhất: P3 trả P0 4 → P2 trả P1 3 → P4 trả P3 2. Dòng cuối của output trên đã kiểm chứng bằng `is_settled` rằng kế hoạch này hợp lệ.

## 9. Code nộp LeetCode (Cách 3 — quay lui có cắt tỉa)

Chuyển sang đúng format: method `minTransfers` nhận `transactions`, tự tính số dư ròng, bỏ số 0, rồi quay lui. Hàm `dfs` đặt **bên trong** method để dùng chung `debts` mà không cần biến toàn cục.

In [16]:
class Solution:
    def minTransfers(self, transactions: List[List[int]]) -> int:
        bal = defaultdict(int)
        for f, t, a in transactions:
            bal[f] += a
            bal[t] -= a
        debts = [b for b in bal.values() if b != 0]
        n = len(debts)

        def dfs(start: int) -> int:
            while start < n and debts[start] == 0:
                start += 1
            if start == n:
                return 0
            best = float("inf")
            tried = set()
            for i in range(start + 1, n):
                if debts[i] * debts[start] < 0 and debts[i] not in tried:
                    tried.add(debts[i])
                    debts[i] += debts[start]
                    best = min(best, 1 + dfs(start + 1))
                    debts[i] -= debts[start]
                    if debts[i] + debts[start] == 0:
                        break
            return best

        return dfs(0)


sol = Solution()
# Ví dụ của đề
assert sol.minTransfers(ex1) == 2
assert sol.minTransfers(ex2) == 1
# Ví dụ xuyên suốt và edge case
assert sol.minTransfers(tricky) == 3
assert sol.minTransfers([[0, 1, 7]]) == 1                              # 1 giao dịch duy nhất
assert sol.minTransfers([[0, 1, 5], [1, 2, 5], [2, 0, 5]]) == 0        # vòng tròn: không ai nợ ai
assert sol.minTransfers([[0, 1, 5], [1, 0, 5]]) == 0                   # đưa rồi trả lại
assert sol.minTransfers([[0, 1, 3], [2, 3, 3]]) == 2                   # hai cặp độc lập
assert sol.minTransfers([[1, 0, 4], [3, 2, 4], [1, 2, 1]]) == 2        # trùng giá trị
print("Tất cả test Solution (quay lui) PASS")

Tất cả test Solution (quay lui) PASS


Kiểm tra thêm trường hợp "nặng" nhất trong phạm vi constraints: 8 giao dịch, ID tới 11, số tiền ngẫu nhiên lớn → thường có 10–12 người lệch và các số dư đôi một khác nhau (cắt tỉa ít tác dụng nhất). Ta tìm trong 300 bộ ngẫu nhiên bộ khiến `dfs` gọi nhiều lần nhất và đo thời gian.

In [17]:
def random_transactions(rng, max_id=12, max_amount=100):
    m = rng.randint(1, 8)
    tr = []
    for _ in range(m):
        f = rng.randrange(max_id)
        t = rng.randrange(max_id)
        while t == f:
            t = rng.randrange(max_id)
        tr.append([f, t, rng.randint(1, max_amount)])
    return tr


rng = random.Random(1)
worst_calls, worst_tr = 0, None
for _ in range(300):
    tr = random_transactions(rng)
    st = {"calls": 0}
    min_transfers_backtrack(nonzero_debts(tr), st)
    if st["calls"] > worst_calls:
        worst_calls, worst_tr = st["calls"], tr

t0 = time.perf_counter()
ans = Solution().minTransfers(worst_tr)
elapsed = time.perf_counter() - t0
print("số người lệch k =", len(nonzero_debts(worst_tr)), "| đáp án =", ans, "| số lần gọi dfs =", worst_calls)
print("chạy dưới 1 giây?", elapsed < 1.0)

số người lệch k = 11 | đáp án = 8 | số lần gọi dfs = 75091
chạy dưới 1 giây? True


## 10. Cách 4 — DP bitmask: tìm **thứ tự xếp hàng** tốt nhất cho cách dây chuyền

### 10.1 Quay lại Cách 1: thử mọi thứ tự (brute force hoán vị)

Ở mục 3.1 ta thấy: dây chuyền theo một thứ tự tốn `k − (số tổng tiền tố bằng 0)` giao dịch. Kết hợp với định lý mục 5:

- Xếp các nhóm tổng 0 của cách chia tối ưu **liền nhau** trong hàng (ví dụ `[3, −3 | 4, −2, −2]`): cuối mỗi nhóm tổng tiền tố bằng 0 → có `G` số 0 → dây chuyền tốn `k − G` = tối ưu.
- Ngược lại, mỗi vị trí có tổng tiền tố bằng 0 "cắt" hàng thành các đoạn tổng 0 → số số 0 không vượt quá `G`.

Vậy: **đáp án = k − (số tổng tiền tố bằng 0 nhiều nhất trên mọi thứ tự)**. Cách vét cạn thẳng thắn là thử mọi hoán vị (permutation — cách sắp thứ tự) bằng `itertools.permutations`:

In [18]:
print("mọi hoán vị của [1, 2, 3]:", list(itertools.permutations([1, 2, 3])))


def chain_cost(order_values):
    # số giao dịch của dây chuyền theo đúng thứ tự này
    zeros = sum(1 for s in itertools.accumulate(order_values) if s == 0)
    return len(order_values) - zeros


def min_transfers_all_orders(debts):
    return min(chain_cost(p) for p in itertools.permutations(debts))


d = nonzero_debts(tricky)
costs = [chain_cost(p) for p in itertools.permutations(d)]
print("tricky: số hoán vị =", len(costs), "| min =", min(costs), "| số hoán vị đạt min =", costs.count(min(costs)))
print("ex1:", min_transfers_all_orders(nonzero_debts(ex1)), "| ex2:", min_transfers_all_orders(nonzero_debts(ex2)))

mọi hoán vị của [1, 2, 3]: [(1, 2, 3), (1, 3, 2), (2, 1, 3), (2, 3, 1), (3, 1, 2), (3, 2, 1)]
tricky: số hoán vị = 120 | min = 3 | số hoán vị đạt min = 24
ex1: 2 | ex2: 1


### 10.2 Hoán vị vướng ở đâu? → chỉ cần nhớ **tập** người đã xếp

Với `k = 12` có `12! = 479 001 600` thứ tự — quá chậm. Nhưng hãy nhìn kỹ: khi đã xếp xong vài người đầu hàng, phần tương lai chỉ phụ thuộc vào **tập hợp** những người đã xếp (vì tổng tiền tố hiện tại = tổng của tập đó), **không** phụ thuộc vào việc họ đứng theo thứ tự nào bên trong.

Ví dụ: `[3, −3, 4]` và `[4, −3, 3]` đều đã xếp tập `{P0, P1, P2}`, tổng tiền tố hiện tại đều là `4`, nên mọi cách xếp tiếp theo cho kết quả giống hệt nhau. Thứ tự bên trong chỉ ảnh hưởng **bao nhiêu số 0 đã gặp** — và ta chỉ cần giữ **số lớn nhất**.

Có `2^k` tập con thay vì `k!` thứ tự: `2^12 = 4096` so với `479 001 600`. Để lưu "một tập con" gọn gàng làm chỉ số mảng, ta dùng **bitmask**.

### 10.3 Nền tảng: bitmask — dùng các bit của một số nguyên làm một tập hợp

Với `n` người, một số nguyên `mask` từ `0` tới `2^n − 1` biểu diễn một tập: **bit thứ `i` bằng 1** nghĩa là người `i` **có** trong tập. Ví dụ `n = 3`: `mask = 5 = 0b101` là tập `{P0, P2}`.

| Thao tác | Code | Ví dụ (`mask = 0b101`) |
|---|---|---|
| tập chỉ gồm người `i` | `1 << i` | `1 << 1 = 0b010` |
| người `i` có trong tập? | `mask >> i & 1` (hoặc `mask & (1 << i)`) | `0b101 >> 2 & 1 = 1` |
| thêm người `i` | `mask \| (1 << i)` | `0b101 \| 0b010 = 0b111` |
| bỏ người `i` (khi chắc chắn có) | `mask ^ (1 << i)` | `0b101 ^ 0b100 = 0b001` |
| tập đầy đủ `n` người | `(1 << n) - 1` | `n = 3` → `0b111 = 7` |

`<<` là dịch trái (nhân 2 mỗi lần), `>>` dịch phải, `&` là AND từng bit, `|` là OR, `^` là XOR (giống thì 0, khác thì 1). `bin(x)` in dạng nhị phân, `format(x, "03b")` in đủ 3 chữ số.

In [19]:
n = 3
for mask in range(1 << n):
    members = [f"P{i}" for i in range(n) if mask >> i & 1]
    print(f"mask={mask}  bit={format(mask, '03b')}  tập={members}")

mask = 0b101
print("có P2?", mask >> 2 & 1, "| thêm P1 ->", format(mask | (1 << 1), "03b"), "| bỏ P2 ->", format(mask ^ (1 << 2), "03b"))

mask=0  bit=000  tập=[]
mask=1  bit=001  tập=['P0']
mask=2  bit=010  tập=['P1']
mask=3  bit=011  tập=['P0', 'P1']
mask=4  bit=100  tập=['P2']
mask=5  bit=101  tập=['P0', 'P2']
mask=6  bit=110  tập=['P1', 'P2']
mask=7  bit=111  tập=['P0', 'P1', 'P2']
có P2? 1 | thêm P1 -> 111 | bỏ P2 -> 001


### 10.4 Tổng số dư của mỗi tập: `total[mask]`

Tổng của một tập = tổng của tập đó **bỏ đi một người bất kỳ** `i` trong tập, cộng `debts[i]`:

```text
total[mask] = total[mask ^ (1 << i)] + debts[i]      (với i là một bit đang bật của mask)
```

Duyệt `mask` tăng dần thì `mask ^ (1 << i)` (nhỏ hơn `mask`) luôn đã được tính trước. Ví dụ với `tricky` `[4, 3, −3, −2, −2]`: `total[0b00110]` (tập `{P1, P2}`) `= total[0b00100] + debts[1] = −3 + 3 = 0`.

In [20]:
d = nonzero_debts(tricky)
n = len(d)
total = [0] * (1 << n)
for mask in range(1, 1 << n):
    i = (mask & -mask).bit_length() - 1        # vị trí bit 1 thấp nhất (xem chú thích bên dưới)
    total[mask] = total[mask ^ (1 << i)] + d[i]

zero_masks = [m for m in range(1, 1 << n) if total[m] == 0]
for m in zero_masks:
    print(f"mask={format(m, '05b')[::-1]} (bit P0..P4)  tập={[f'P{i}' for i in range(n) if m >> i & 1]}  tổng=0")

mask=01100 (bit P0..P4)  tập=['P1', 'P2']  tổng=0
mask=10011 (bit P0..P4)  tập=['P0', 'P3', 'P4']  tổng=0
mask=11111 (bit P0..P4)  tập=['P0', 'P1', 'P2', 'P3', 'P4']  tổng=0


*Chú thích:* `mask & -mask` giữ lại đúng **bit 1 thấp nhất** của `mask` (ví dụ `0b0110 & -0b0110 = 0b0010`), và `.bit_length() − 1` đổi nó thành vị trí bit. Đây là mẹo quen thuộc, nhưng không bắt buộc — dùng vòng `for i in range(n)` tìm bit bật đầu tiên cũng được. Ở output trên, chuỗi bit được **in ngược** để ký tự đầu tiên ứng với P0.

Đúng như ta tìm bằng tay: chỉ có 3 tập tổng 0 là `{P1, P2}`, `{P0, P3, P4}` và cả 5 người.

### 10.5 DP trên tập: `dp[mask]`

**Định nghĩa:** `dp[mask]` = số **tổng tiền tố bằng 0 nhiều nhất** khi xếp hàng **đúng các người trong `mask`** theo một thứ tự nào đó.

**Công thức:** người đứng **cuối** hàng là một người `i` nào đó trong `mask`. Trước họ là một cách xếp của `mask ^ (1 << i)`. Tổng tiền tố tại vị trí cuối chính là `total[mask]`. Nên

```text
dp[0]    = 0
dp[mask] = max( dp[mask ^ (1 << i)]  với mọi i trong mask )  +  (1 nếu total[mask] == 0, ngược lại 0)
```

**Đáp án:** `n − dp[(1 << n) − 1]`.

Ví dụ tay với `tricky`: `dp[{P1,P2}] = 1` (xếp `3, −3`), `dp[{P0,P1,P2,P3,P4}] = dp[{P0,P1,P2,P3}] + 1` nếu người cuối là P4, và `dp[{P0,P1,P2,P3}]` có thể bằng 1 (xếp `3, −3, 4, −2`) → `dp[full] = 2` → đáp án `5 − 2 = 3`.

Đây chính là bài toán "chia thành nhiều nhóm tổng 0 nhất" của mục 5.2, nhưng giải trong `O(2^n · n)` thay vì gần `3^n`.

In [21]:
def min_transfers_bitmask(debts):
    n = len(debts)
    full = (1 << n) - 1
    total = [0] * (1 << n)
    dp = [0] * (1 << n)
    for mask in range(1, 1 << n):
        best = 0
        for i in range(n):
            if mask >> i & 1:
                prev = mask ^ (1 << i)
                total[mask] = total[prev] + debts[i]   # tính lại cũng ra cùng một giá trị
                best = max(best, dp[prev])
        dp[mask] = best + (1 if total[mask] == 0 else 0)
    return n - dp[full]


for name, tr in [("ex1", ex1), ("ex2", ex2), ("tricky", tricky)]:
    print(f"{name:7s}: {min_transfers_bitmask(nonzero_debts(tr))}")
print("không ai lệch (debts rỗng):", min_transfers_bitmask([]))

ex1    : 2
ex2    : 1
tricky : 3
không ai lệch (debts rỗng): 0


### 10.6 Trace DP bitmask và truy vết thứ tự xếp hàng

Hàm `trace_bitmask` in các tập có tổng 0 cùng giá trị `dp`, rồi **truy vết** (đi ngược từ `full`): ở mỗi bước chọn một người cuối `i` sao cho `dp[mask] == dp[mask ^ (1 << i)] + (total[mask] == 0)`. Đảo ngược danh sách đó ta có thứ tự xếp hàng tối ưu; đưa vào `settle_chain` là ra **kế hoạch thanh toán cụ thể** với đúng `k − G` giao dịch.

Dự đoán:

| Tập (mask) | `total` | `dp` | Ý nghĩa |
|---|---:|---:|---|
| `{P1, P2}` | 0 | 1 | xếp `3, −3` có 1 số 0 |
| `{P0, P3, P4}` | 0 | 1 | xếp `4, −2, −2` có 1 số 0 |
| `{P0..P4}` | 0 | 2 | ghép hai nhóm liền nhau |

In [22]:
def trace_bitmask(transactions):
    bal = net_balances(transactions)
    people = [p for p, b in bal.items() if b != 0]
    debts = [bal[p] for p in people]
    n = len(debts)
    full = (1 << n) - 1
    total = [0] * (1 << n)
    dp = [0] * (1 << n)
    for mask in range(1, 1 << n):
        for i in range(n):
            if mask >> i & 1:
                total[mask] = total[mask ^ (1 << i)] + debts[i]
                dp[mask] = max(dp[mask], dp[mask ^ (1 << i)])
        dp[mask] += total[mask] == 0           # True cộng như 1, False như 0
    for mask in range(1, 1 << n):
        if total[mask] == 0:
            print(f"tập {[f'P{people[i]}' for i in range(n) if mask >> i & 1]}: total=0, dp={dp[mask]}")
    # truy vết thứ tự
    order, mask = [], full
    while mask:
        for i in range(n):
            if mask >> i & 1 and dp[mask] == dp[mask ^ (1 << i)] + (total[mask] == 0):
                order.append(people[i])
                mask ^= 1 << i
                break
    order.reverse()
    plan = settle_chain(bal, order)
    print("thứ tự tối ưu:", order, "| số dư:", [bal[p] for p in order],
          "| tổng tiền tố:", list(itertools.accumulate(bal[p] for p in order)))
    print("kế hoạch:", plan, "| hợp lệ?", is_settled(transactions, plan), "| số giao dịch =", n - dp[full])
    return plan


_ = trace_bitmask(tricky)

tập ['P1', 'P2']: total=0, dp=1
tập ['P0', 'P3', 'P4']: total=0, dp=1
tập ['P0', 'P1', 'P2', 'P3', 'P4']: total=0, dp=2
thứ tự tối ưu: [2, 1, 4, 3, 0] | số dư: [-3, 3, -2, -2, 4] | tổng tiền tố: [-3, 0, -2, -4, 0]
kế hoạch: [[2, 1, 3], [4, 3, 2], [3, 0, 4]] | hợp lệ? True | số giao dịch = 3


### 10.7 Bản LeetCode của Cách 4

Bản này dùng mẹo bit thấp nhất để tính `total` (mỗi `mask` chỉ một phép cộng) và vòng `for i` để lấy `max`. Vẫn là `O(2^n · n)`, với `n <= 12` chỉ khoảng `4096 · 12 ≈ 49 000` bước.

In [23]:
class SolutionBitmask:
    def minTransfers(self, transactions: List[List[int]]) -> int:
        bal = defaultdict(int)
        for f, t, a in transactions:
            bal[f] += a
            bal[t] -= a
        debts = [b for b in bal.values() if b != 0]
        n = len(debts)
        total = [0] * (1 << n)
        dp = [0] * (1 << n)
        for mask in range(1, 1 << n):
            low = (mask & -mask).bit_length() - 1
            total[mask] = total[mask ^ (1 << low)] + debts[low]
            best = 0
            for i in range(n):
                if mask >> i & 1:
                    best = max(best, dp[mask ^ (1 << i)])
            dp[mask] = best + (total[mask] == 0)
        return n - dp[(1 << n) - 1]


solb = SolutionBitmask()
assert solb.minTransfers(ex1) == 2
assert solb.minTransfers(ex2) == 1
assert solb.minTransfers(tricky) == 3
assert solb.minTransfers([[0, 1, 5], [1, 2, 5], [2, 0, 5]]) == 0
assert solb.minTransfers([[0, 1, 3], [2, 3, 3]]) == 2
assert solb.minTransfers(worst_tr) == Solution().minTransfers(worst_tr)
print("Tất cả test SolutionBitmask PASS")

Tất cả test SolutionBitmask PASS


## 11. Đối chiếu chéo các cách trên dữ liệu ngẫu nhiên

Ta sinh nhiều bộ `transactions` ngẫu nhiên **đúng constraints** (1–8 giao dịch, ID `< 12`, số tiền 1–100) với `random.Random(2024)` cố định. Vì số tiền lớn ngẫu nhiên hiếm khi tạo ra nhóm tổng 0 nhỏ (đáp án thường chỉ là `k − 1`), ta trộn thêm các bộ có **ít người** và **số tiền nhỏ** để có nhiều trường hợp "tiết kiệm được giao dịch".

Kiểm tra:
1. Quay lui có cắt tỉa = quay lui thuần = DP bitmask = vét cạn chia nhóm (mục 5.2); và = hoán vị (mục 10.1) khi `k <= 7`.
2. Dây chuyền và tham lam luôn **hợp lệ** (`is_settled`) và **không bao giờ ít hơn** tối ưu.
3. Kế hoạch truy vết từ DP bitmask có đúng số giao dịch tối ưu và hợp lệ.

In [24]:
def bitmask_plan(transactions):
    # như trace_bitmask nhưng không in gì: trả về kế hoạch tối ưu
    bal = net_balances(transactions)
    people = [p for p, b in bal.items() if b != 0]
    debts = [bal[p] for p in people]
    n = len(debts)
    total, dp = [0] * (1 << n), [0] * (1 << n)
    for mask in range(1, 1 << n):
        for i in range(n):
            if mask >> i & 1:
                total[mask] = total[mask ^ (1 << i)] + debts[i]
                dp[mask] = max(dp[mask], dp[mask ^ (1 << i)])
        dp[mask] += total[mask] == 0
    order, mask = [], (1 << n) - 1
    while mask:
        for i in range(n):
            if mask >> i & 1 and dp[mask] == dp[mask ^ (1 << i)] + (total[mask] == 0):
                order.append(people[i]); mask ^= 1 << i
                break
    return settle_chain(bal, order[::-1])


rng = random.Random(2024)
settings = [(12, 100), (6, 5), (4, 3), (12, 4)]
stats = {"tests": 0, "saved": 0, "greedy_worse": 0, "chain_worse": 0}
for it in range(1200):
    max_id, max_amount = settings[it % len(settings)]
    tr = random_transactions(rng, max_id, max_amount)
    d = nonzero_debts(tr)
    opt = min_transfers_backtrack(d)
    assert opt == min_transfers_backtrack_plain(d) == min_transfers_bitmask(d) == min_transfers_partition_bruteforce(d), tr
    assert opt == Solution().minTransfers(tr) == SolutionBitmask().minTransfers(tr), tr
    if len(d) <= 7:
        assert opt == min_transfers_all_orders(d), tr
    bal = net_balances(tr)
    g, c, p = settle_greedy(bal), settle_chain(bal), bitmask_plan(tr)
    assert is_settled(tr, g) and is_settled(tr, c) and is_settled(tr, p), tr
    assert len(g) >= opt and len(c) >= opt and len(p) == opt, tr
    stats["tests"] += 1
    stats["saved"] += opt < max(len(d) - 1, 0)
    stats["greedy_worse"] += len(g) > opt
    stats["chain_worse"] += len(c) > opt
print(stats)
print("-> mọi cách tối ưu khớp nhau; tham lam và dây chuyền hợp lệ nhưng có lúc tốn hơn tối ưu.")

{'tests': 1200, 'saved': 658, 'greedy_worse': 108, 'chain_worse': 546}
-> mọi cách tối ưu khớp nhau; tham lam và dây chuyền hợp lệ nhưng có lúc tốn hơn tối ưu.


## 12. Vì sao các cách đúng? (tóm tắt)

- **Rút gọn về số dư ròng:** kế hoạch hợp lệ ⇔ cộng thêm nó vào thì ai cũng về 0; điều kiện này chỉ phụ thuộc số dư ròng. Người số dư 0 không cần tham gia.
- **Định lý `k − G`:**
  - *Đủ*: mỗi nhóm tổng 0 có `s` người được giải bằng dây chuyền `s − 1` giao dịch.
  - *Cần*: mỗi giao dịch nối tối đa hai cụm; mỗi cụm sau cùng phải có tổng 0; nên `T >= k − (số cụm) >= k − G`.
- **Cách 3 (quay lui):**
  - *Bất biến*: khi vào `dfs(start)`, tổng `debts[start:]` bằng 0 và mọi người trước `start` đã về 0 → luôn có người trái dấu để ghép, và mỗi nhánh là một kế hoạch hợp lệ (nên `dfs >=` tối ưu).
  - *Không bỏ sót*: trong nhóm tổng 0 chứa `start` luôn có người trái dấu; chuyển hết cho họ giữ nhóm tổng 0 và bớt một người → nhánh này đạt `s − 1` cho nhóm, tổng cộng `k − G` (nên `dfs <=` tối ưu).
  - *Cắt tỉa 1*: hai lựa chọn có cùng giá trị `debts[i]` tạo ra cùng multiset → cùng kết quả.
  - *Cắt tỉa 2*: cặp triệt tiêu `{a, b}` luôn nằm trong **một** cách chia tối ưu (lập luận đổi chỗ ở mục 7.4).
- **Cách 4 (DP bitmask):** số tổng tiền tố bằng 0 nhiều nhất trên mọi thứ tự = `G`; `dp[mask]` chỉ phụ thuộc tập đã xếp vì tổng tiền tố hiện tại = `total[mask]`, và người cuối hàng có thể là bất kỳ ai trong `mask` → công thức `max` xét đủ mọi khả năng.
- **Cách 1, 2** luôn hợp lệ (mỗi bước có người về 0, tổng luôn bằng 0) nhưng không tối ưu — phản ví dụ `tricky`.

## 13. Độ phức tạp và so sánh các cách

Ký hiệu: `m` = số giao dịch (`<= 8`), `n` = số người có số dư khác 0 (`<= 12`).

| Cách | Ý tưởng | Time | Space | Tối ưu? | Ghi chú |
|---|---|---|---|:---:|---|
| Số dư ròng | `defaultdict` cộng dồn | `O(m)` | `O(n)` | — | bước chuẩn bị của mọi cách |
| 1. Dây chuyền | chuyền phần lệch theo hàng | `O(n)` | `O(n)` | ❌ | luôn `<= n − 1` |
| 2. Tham lam | nợ lớn nhất ↔ được nợ lớn nhất | `O(n^2)` | `O(n)` | ❌ | phản ví dụ `tricky` |
| Vét cạn chia nhóm | thử mọi tập con ghép với phần tử đầu | `≈ O(3^n)` | `O(n)` đệ quy | ✅ | dùng làm trọng tài |
| Hoán vị | thử mọi thứ tự dây chuyền | `O(n! · n)` | `O(n)` | ✅ | chỉ để dẫn dắt tới bitmask |
| **3. Quay lui + cắt tỉa** | người đầu tiên ghép với người trái dấu | worst `O((n−1)!)`, thực tế rất nhanh | `O(n)` đệ quy | ✅ | **cách chính**, kinh điển |
| **4. DP bitmask** | `dp[mask]` = số tổng tiền tố 0 nhiều nhất | `O(2^n · n)` | `O(2^n)` | ✅ | ổn định, dễ chứng minh |

Giải thích các con số:

- Quay lui: ở tầng `start` có tối đa `n − 1 − start` lựa chọn, độ sâu tối đa `n − 1` → cận trên `(n − 1)!` (với `n = 12` là khoảng 40 triệu), nhưng nhờ cắt tỉa và nhờ các nhánh triệt tiêu sớm, số lần gọi thực tế nhỏ hơn nhiều (mục 9 đo được chạy dưới 1 giây cho bộ nặng nhất tìm được).
- Bitmask: `2^12 · 12 ≈ 49 000` bước, gần như tức thì và **không phụ thuộc** dữ liệu — đó là ưu điểm lớn so với quay lui, đổi lại tốn mảng `2^n` phần tử và chỉ dùng được khi `n` nhỏ (khoảng `<= 20`).

## 14. Lỗi thường gặp

### Lỗi 1: Quên bỏ qua người đã về 0 (thiếu vòng `while`)

```python
def dfs(start):
    if start == n:            # chỉ kiểm tra start == n
        return 0
    ...                       # nếu debts[start] == 0 thì không có ai "trái dấu" với 0
```

**Vì sao sai:** sau khi P1 (−10) ở Example 1 nhận phần của P0 và chuyển cho P2, P2 về đúng 0. Khi gọi `dfs` tại P2, điều kiện `debts[i] * 0 < 0` không bao giờ đúng → vòng `for` không chạy → trả `inf`. **Sửa:** luôn có `while start < n and debts[start] == 0: start += 1` trước base case.

### Lỗi 2: Quên hoàn tác `debts[i] -= debts[start]`

**Vì sao sai:** nhánh trước để lại "rác" trong `debts`, nhánh sau bắt đầu từ trạng thái sai (giống hệt ví dụ quên `path.pop()` ở mục 6). Kết quả có thể **sai** chứ không chỉ chậm. **Sửa:** mỗi lần `+=` phải có đúng một lần `-=` ngay sau lời gọi đệ quy.

### Lỗi 3: Gọi `dfs(i + 1)` thay vì `dfs(start + 1)`

**Vì sao sai:** người vừa được ghép là `i`, nhưng những người **giữa** `start` và `i` vẫn chưa xong! Gọi `dfs(i + 1)` bỏ qua họ, trả về số **quá nhỏ** (thậm chí `inf` khi người `i` còn lệch mà bị bỏ qua). **Sửa:** người tiếp theo cần xử lý luôn là `start + 1`; người `i` vẫn nằm trong phần còn lại với số dư mới.

In [25]:
def bt_no_skip(debts):                       # Lỗi 1
    debts = list(debts); n = len(debts)
    def dfs(start):
        if start == n:
            return 0
        best = float("inf")
        for i in range(start + 1, n):
            if debts[i] * debts[start] < 0:
                debts[i] += debts[start]; best = min(best, 1 + dfs(start + 1)); debts[i] -= debts[start]
        return best
    return dfs(0)


def bt_no_undo(debts):                       # Lỗi 2
    debts = list(debts); n = len(debts)
    def dfs(start):
        while start < n and debts[start] == 0:
            start += 1
        if start == n:
            return 0
        best = float("inf")
        for i in range(start + 1, n):
            if debts[i] * debts[start] < 0:
                debts[i] += debts[start]; best = min(best, 1 + dfs(start + 1))   # quên -=
        return best
    return dfs(0)


def bt_wrong_next(debts):                    # Lỗi 3
    debts = list(debts); n = len(debts)
    def dfs(start):
        while start < n and debts[start] == 0:
            start += 1
        if start == n:
            return 0
        best = float("inf")
        for i in range(start + 1, n):
            if debts[i] * debts[start] < 0:
                debts[i] += debts[start]; best = min(best, 1 + dfs(i + 1)); debts[i] -= debts[start]
        return best
    return dfs(0)


d1, dt = nonzero_debts(ex1), nonzero_debts(tricky)
print("Lỗi 1 (thiếu while) ex1   :", bt_no_skip(d1), "   đúng:", min_transfers_backtrack(d1))
print("Lỗi 2 (quên hoàn tác) tricky:", bt_no_undo(dt), "     đúng:", min_transfers_backtrack(dt))
print("Lỗi 3 (dfs(i+1)) tricky     :", bt_wrong_next(dt), "     đúng:", min_transfers_backtrack(dt))
print("Lỗi 3 (dfs(i+1)) ex1        :", bt_wrong_next(d1), "   đúng:", min_transfers_backtrack(d1))

Lỗi 1 (thiếu while) ex1   : inf    đúng: 2
Lỗi 2 (quên hoàn tác) tricky: 4      đúng: 3
Lỗi 3 (dfs(i+1)) tricky     : 1      đúng: 3
Lỗi 3 (dfs(i+1)) ex1        : inf    đúng: 2


### Lỗi 4: Tạo `tried` **một lần** cho cả hàm (dùng chung giữa các tầng)

```python
tried = set()            # đặt NGOÀI dfs -> mọi tầng dùng chung
def dfs(start):
    ...
```

**Vì sao sai:** cắt tỉa 1 chỉ đúng **trong cùng một tầng** (cùng `start`, cùng trạng thái). Nếu dùng chung, một giá trị đã thử ở tầng khác (trạng thái khác!) bị coi là "đã thử" → bỏ sót nhánh, thậm chí bỏ hết mọi nhánh và trả `inf`. **Sửa:** `tried = set()` là dòng đầu **bên trong** vòng gọi `dfs`, trước vòng `for`.

### Lỗi 5: `break` sai chỗ — vô tình biến quay lui thành tham lam

Viết `return` (hoặc `break`) ngay sau nhánh đầu tiên — "chắc người trái dấu đầu tiên là được" — thì chỉ còn đúng **một** nhánh: đó là một thuật toán tham lam trá hình, sai trên `tricky`. Cắt tỉa 2 chỉ được `break` **khi cặp triệt tiêu đúng** (`debts[i] + debts[start] == 0`, kiểm tra **sau** khi hoàn tác).

### Lỗi 6 (không sai nhưng chậm): ghép cả với người **cùng dấu**

Nếu bỏ điều kiện `debts[i] * debts[start] < 0` (chỉ cần `debts[i] != 0`), mỗi nhánh vẫn là kế hoạch hợp lệ nên đáp án vẫn đúng — nhưng các nhánh "cùng dấu" làm người `i` lệch **nhiều hơn**, không bao giờ cần thiết (mục 7.1 đã chỉ ra luôn có lựa chọn trái dấu tối ưu), và làm số lần gọi tăng vọt.

In [26]:
def bt_global_tried(debts):                  # Lỗi 4
    debts = list(debts); n = len(debts)
    tried = set()                            # sai: dùng chung
    def dfs(start):
        while start < n and debts[start] == 0:
            start += 1
        if start == n:
            return 0
        best = float("inf")
        for i in range(start + 1, n):
            if debts[i] * debts[start] < 0 and debts[i] not in tried:
                tried.add(debts[i])
                debts[i] += debts[start]; best = min(best, 1 + dfs(start + 1)); debts[i] -= debts[start]
        return best
    return dfs(0)


def bt_first_only(debts):                    # Lỗi 5
    debts = list(debts); n = len(debts)
    def dfs(start):
        while start < n and debts[start] == 0:
            start += 1
        if start == n:
            return 0
        for i in range(start + 1, n):
            if debts[i] * debts[start] < 0:
                debts[i] += debts[start]; r = 1 + dfs(start + 1); debts[i] -= debts[start]
                return r                     # sai: chỉ thử nhánh đầu tiên
    return dfs(0)


def bt_same_sign_too(debts, stats):          # Lỗi 6
    debts = list(debts); n = len(debts)
    def dfs(start):
        stats["calls"] += 1
        while start < n and debts[start] == 0:
            start += 1
        if start == n:
            return 0
        best = float("inf")
        for i in range(start + 1, n):
            if debts[i] != 0:                # cả cùng dấu
                debts[i] += debts[start]; best = min(best, 1 + dfs(start + 1)); debts[i] -= debts[start]
        return best
    return dfs(0)


print("Lỗi 4 (tried chung) tricky :", bt_global_tried(dt), "  đúng:", min_transfers_backtrack(dt))
print("Lỗi 5 (chỉ nhánh đầu) tricky:", bt_first_only(dt), "    đúng:", min_transfers_backtrack(dt))
s_same, s_opp = {"calls": 0}, {"calls": 0}
r_same = bt_same_sign_too([2, 2, 2, 2, -1, -1, -1, -1, -4], s_same)
r_opp = min_transfers_backtrack_plain([2, 2, 2, 2, -1, -1, -1, -1, -4], s_opp)
print(f"Lỗi 6: đáp án {r_same} = {r_opp} (vẫn đúng) nhưng gọi dfs {s_same['calls']} lần so với {s_opp['calls']} lần")

Lỗi 4 (tried chung) tricky : inf   đúng: 3
Lỗi 5 (chỉ nhánh đầu) tricky: 4     đúng: 3
Lỗi 6: đáp án 6 = 6 (vẫn đúng) nhưng gọi dfs 75607 lần so với 1467 lần


### Lỗi 7: DP bitmask — cộng `+1` cho **mọi** mask, hoặc trả nhầm `dp[full]`

```python
dp[mask] = best + 1                  # sai: quên điều kiện total[mask] == 0
return dp[full]                      # sai: dp[full] là số NHÓM, không phải số giao dịch
```

**Vì sao sai:** `dp` đếm số tổng tiền tố **bằng 0**, nên chỉ được `+1` khi `total[mask] == 0`; nếu `+1` mọi lúc thì `dp[full] = n` và đáp án luôn thành `0`. Còn `dp[full]` là `G` (số nhóm), đáp án là `n − G`. **Sửa:** `dp[mask] = best + (total[mask] == 0)` và `return n - dp[full]`.

### Lỗi 8: Hiểu sai đề

- *"Phải hoàn trả theo đúng các cặp đã giao dịch"*: sai — giao dịch mới có thể giữa **bất kỳ** hai người (Example 1: P1 trả thẳng P2 dù hai người chưa từng giao dịch).
- *"Đáp án ≤ số giao dịch gốc nên cứ trả `len(transactions)`"*: sai — ví dụ vòng tròn `[[0,1,5],[1,2,5],[2,0,5]]` có đáp án `0`.
- *Đổi quy ước dấu* (`bal[f] -= a`, `bal[t] += a`): **không** phải lỗi — mọi số dư đổi dấu cùng lúc, nhóm tổng 0 vẫn là nhóm tổng 0, đáp án không đổi. Chỉ cần **nhất quán** trong cả bài.

In [27]:
def bitmask_bug_always_plus(debts):          # Lỗi 7a
    n = len(debts); dp = [0] * (1 << n)
    for mask in range(1, 1 << n):
        dp[mask] = max(dp[mask ^ (1 << i)] for i in range(n) if mask >> i & 1) + 1
    return n - dp[(1 << n) - 1]


def bitmask_bug_return_dp(debts):            # Lỗi 7b
    n = len(debts); total = [0] * (1 << n); dp = [0] * (1 << n)
    for mask in range(1, 1 << n):
        for i in range(n):
            if mask >> i & 1:
                total[mask] = total[mask ^ (1 << i)] + debts[i]
                dp[mask] = max(dp[mask], dp[mask ^ (1 << i)])
        dp[mask] += total[mask] == 0
    return dp[(1 << n) - 1]


print("Lỗi 7a tricky:", bitmask_bug_always_plus(dt), "| Lỗi 7b tricky:", bitmask_bug_return_dp(dt), "| đúng:", min_transfers_bitmask(dt))

flipped = [[t, f, a] for f, t, a in tricky]            # đảo chiều mọi giao dịch = đổi dấu mọi số dư
print("Đổi dấu toàn bộ:", nonzero_debts(flipped), "-> đáp án", Solution().minTransfers(flipped), "(không đổi)")
print("Vòng tròn:", Solution().minTransfers([[0, 1, 5], [1, 2, 5], [2, 0, 5]]), "giao dịch")

Lỗi 7a tricky: 0 | Lỗi 7b tricky: 2 | đúng: 3
Đổi dấu toàn bộ: [-4, -3, 3, 2, 2] -> đáp án 3 (không đổi)
Vòng tròn: 0 giao dịch


## 15. Mẹo nhận dạng pattern và bài liên quan

### Dấu hiệu nhận ra bài "chia tập nhỏ thành các nhóm" (backtracking / DP bitmask)

- **Kích thước rất nhỏ** trong constraints: số phần tử thực sự cần xét `<= 12…20` (ở đây `transactions.length <= 8`, ID `< 12`). Đề Hard mà `n` nhỏ cỡ này gần như là lời mời dùng `2^n` hoặc quay lui.
- Câu hỏi về **cách gom / chia / ghép** các phần tử thành nhóm thỏa điều kiện về **tổng** (tổng 0, tổng bằng nhau, tổng không vượt quá…).
- Tham lam "to nhất trước" nghe hợp lý nhưng có phản ví dụ nhỏ → cần vét cạn có tổ chức.
- Bài toán có thể **rút gọn** trước khi tìm kiếm: ở đây từ danh sách giao dịch → danh sách số dư (bước này thường là nửa lời giải).
- Bài toán tổng quát được biết là **NP-khó** (không có thuật toán đa thức đã biết), nên đề mới giới hạn input nhỏ — gặp tình huống này đừng cố tìm công thức `O(n)`.

### Hai "khuôn" cần thuộc

| Khuôn | Khi nào | Mẫu |
|---|---|---|
| Quay lui "phần tử đầu tiên chưa xử lý" | chọn bạn ghép cho phần tử đầu, đệ quy phần còn lại | `while debts[start]==0: start+=1` → thử `i` → hoàn tác → cắt tỉa trùng |
| DP trên tập con | trạng thái = "tập đã dùng", thứ tự bên trong không quan trọng | `for mask: for i in mask: dp[mask] = f(dp[mask ^ (1<<i)])` |

### Bài liên quan

| Bài | Liên hệ |
|---|---|
| 698. Partition to K Equal Sum Subsets | chia thành `k` nhóm tổng bằng nhau — DP bitmask với "phần dư hiện tại" (Bài 6) |
| 473. Matchsticks to Square | trường hợp `k = 4` của bài trên, quay lui có cắt tỉa |
| 2305. Fair Distribution of Cookies | chia cho `k` em, tối thiểu tổng lớn nhất — quay lui / DP trên tập con |
| 1723. Find Minimum Time to Finish All Jobs | như trên, cắt tỉa giá trị trùng ở cùng tầng |
| 1986. Minimum Number of Work Sessions to Finish the Tasks | DP bitmask "số phiên + thời gian đang dùng" |
| 1655. Distribute Repeating Integers | DP trên tập con khách hàng |
| Splitwise / "debt simplification" | ứng dụng thực tế; app thường dùng tham lam (nhanh, không tối ưu) vì nhóm lớn |

## 16. Bài tập vận dụng

Làm theo thứ tự, tự làm xong rồi mới xem **Gợi ý** (mục 17) và **Đáp án** (mục 18).

### Bài 1 — Đoán output (không chạy code)

1. `net_balances([[0, 1, 5], [1, 2, 5], [2, 0, 5]])` trả về gì? `Solution().minTransfers(...)` của bộ này bằng bao nhiêu?
2. `Solution().minTransfers([[0, 1, 7]])` bằng bao nhiêu?
3. Với `[[0, 1, 3], [0, 2, 3], [3, 0, 6]]`: `nonzero_debts` là gì, và đáp án bằng bao nhiêu?
4. `chain_cost([3, -3, 4, -2, -2])` bằng bao nhiêu? Còn `chain_cost([4, -2, 3, -2, -3])`?
5. Với `debts = [4, 3, -3, -2, -2]`, `mask = 0b00110` (bit 0 bên phải) là tập nào và `total[mask]` bằng bao nhiêu?

### Bài 2 — Tính tay

Cho `debts = [6, -2, -4, 5, -5, 1, -1]` (người P0 … P6).

a) Chia thành nhiều nhóm tổng 0 nhất. `G` bằng bao nhiêu? Đáp án tối ưu bằng bao nhiêu?

b) Dây chuyền theo đúng thứ tự P0 … P6 tốn bao nhiêu giao dịch? (Gợi ý: viết tổng tiền tố.)

c) Chạy tay thuật toán tham lam (mục 4, khi hòa thì lấy người có ID nhỏ hơn). Nó tốn bao nhiêu giao dịch?

### Bài 3 — Điền TODO: quay lui có cắt tỉa

Viết lại lời giải chính mà **không nhìn** mục 7–9. Hàm nhận `transactions` (không phải `debts`).

In [28]:
def min_transfers_exercise(transactions):
    # TODO 1: tính số dư ròng bằng defaultdict(int): bal[f] += a, bal[t] -= a
    # TODO 2: debts = các số dư khác 0, n = len(debts)
    # TODO 3: viết dfs(start):
    #   - bỏ qua người đã về 0 (while)
    #   - base case start == n -> 0
    #   - tried = set() MỚI cho tầng này
    #   - với i > start trái dấu và chưa thử: CHỌN (debts[i] += debts[start]),
    #     ĐỆ QUY 1 + dfs(start + 1), HOÀN TÁC (debts[i] -= debts[start])
    #   - nếu debts[i] + debts[start] == 0 thì break
    # TODO 4: return dfs(0)
    pass


# Tự làm xong rồi mới mở test:
# assert min_transfers_exercise(ex1) == 2
# assert min_transfers_exercise(ex2) == 1
# assert min_transfers_exercise(tricky) == 3
# assert min_transfers_exercise([[0, 1, 5], [1, 2, 5], [2, 0, 5]]) == 0

### Bài 4 — Tự viết: DP bitmask

Viết `min_transfers_bitmask_exercise(transactions)` theo Cách 4 mà không nhìn mục 10. Yêu cầu: dùng mảng `total` và `dp` kích thước `1 << n`, trả `n - dp[full]`.

In [29]:
def min_transfers_bitmask_exercise(transactions):
    # TODO: số dư ròng -> debts khác 0 -> total[mask], dp[mask] -> n - dp[full]
    raise NotImplementedError


# Tự làm xong rồi mới mở test:
# assert min_transfers_bitmask_exercise(ex1) == 2
# assert min_transfers_bitmask_exercise(ex2) == 1
# assert min_transfers_bitmask_exercise(tricky) == 3
# assert min_transfers_bitmask_exercise([[0, 1, 3], [2, 3, 3]]) == 2

### Bài 5 — Biến thể: trả về **chính kế hoạch** thanh toán tối ưu

Đề phỏng vấn hay hỏi tiếp: "Không chỉ số lượng — hãy in ra **ai trả ai bao nhiêu**". Viết `optimal_plan(transactions)` trả về list `[người_trả, người_nhận, số_tiền]` (ID thật, không phải chỉ số trong `debts`) sao cho:

- `is_settled(transactions, plan)` là `True`;
- `len(plan) == Solution().minTransfers(transactions)`.

Yêu cầu: dùng **quay lui** (sửa Cách 3 để `dfs` trả về list giao dịch thay vì số), không dùng `bitmask_plan`.

In [30]:
def optimal_plan(transactions):
    # TODO 1: bal = net_balances(transactions); people = ID có số dư khác 0; debts tương ứng
    # TODO 2: dfs(start) trả về LIST giao dịch tối ưu cho phần từ start trở đi
    #         (base case: list rỗng). Khi ghép start với i, giao dịch có chiều tùy dấu debts[start].
    # TODO 3: giữ nhánh có list ngắn nhất; nhớ hoàn tác và hai cắt tỉa
    pass


# Tự làm xong rồi mới mở test:
# for tr in (ex1, ex2, tricky):
#     plan = optimal_plan(tr)
#     assert is_settled(tr, plan) and len(plan) == Solution().minTransfers(tr)

### Bài 6 — Mini challenge: LeetCode 698. Partition to K Equal Sum Subsets

> Cho mảng `nums` và số `k`, trả về `True` nếu chia được `nums` thành `k` tập con khác rỗng có **tổng bằng nhau**. Constraints: `1 <= k <= nums.length <= 16`, `1 <= nums[i] <= 10^4`, mỗi phần tử có tần suất trong khoảng `[1, 4]`.

Ví dụ: `nums = [4, 3, 2, 3, 5, 2, 1], k = 4` → `True` (`(5), (1,4), (2,3), (2,3)`); `nums = [1, 2, 3, 4], k = 3` → `False`.

Dùng ý tưởng "xếp hàng" của Cách 4: xếp các phần tử lần lượt vào các "giỏ" có sức chứa `target = sum(nums) // k`, giỏ đầy thì sang giỏ mới. Trạng thái chỉ cần **tập đã xếp** (`mask`) — vì phần đã đổ vào giỏ hiện tại luôn là `total[mask] % target`.

In [31]:
def can_partition_k_subsets(nums, k):
    # TODO 1: nếu sum(nums) % k != 0 -> False; target = sum(nums) // k
    # TODO 2: ok[mask] = True nếu xếp được tập mask theo luật "không tràn giỏ"
    # TODO 3: chuyển trạng thái mask -> mask | (1 << i) khi (total[mask] % target) + nums[i] <= target
    # TODO 4: trả ok[full]
    pass


# Tự làm xong rồi mới mở test:
# assert can_partition_k_subsets([4, 3, 2, 3, 5, 2, 1], 4) is True
# assert can_partition_k_subsets([1, 2, 3, 4], 3) is False
# assert can_partition_k_subsets([2, 2, 2, 2, 3, 4, 5], 4) is False

## 17. Gợi ý theo mức

**Bài 1**
- Mức 1: tính số dư từng người trước, mọi câu hỏi khác suy ra từ đó.
- Mức 2: câu 3 — P0 đưa 3 + 3 và nhận 6; câu 4 — viết tổng tiền tố rồi đếm số 0.
- Mức 3: câu 5 — bit 1 và bit 2 đang bật → người chỉ số 1 và 2.

**Bài 2**
- Mức 1: tìm các **cặp** triệt tiêu trước (`5, −5` và `1, −1`), rồi xem phần còn lại.
- Mức 2: `6 − 2 − 4 = 0`. Tổng tiền tố theo thứ tự gốc: `6, 4, 0, 5, 0, 1, 0`.
- Mức 3: tham lam bắt đầu bằng `P4 trả P0 5` (lớn nhất `+6` gặp nhỏ nhất `−5`) — chính là "phá cặp" `(5, −5)`.

**Bài 3**
- Mức 1: chép khung hàm `dfs` của Cách 3 thuần, rồi thêm hai cắt tỉa.
- Mức 2: `tried = set()` đặt **trong** `dfs`, ngay trước vòng `for`; điều kiện: `debts[i] * debts[start] < 0 and debts[i] not in tried`.
- Mức 3: `if debts[i] + debts[start] == 0: break` đặt **sau** dòng hoàn tác.

**Bài 4**
- Mức 1: hai mảng `total`, `dp`, duyệt `mask` từ `1` tới `(1 << n) - 1`.
- Mức 2: `for i in range(n): if mask >> i & 1: prev = mask ^ (1 << i)`.
- Mức 3: `dp[mask] = max(dp[prev]...) + (total[mask] == 0)`; `return n - dp[(1 << n) - 1]`.

**Bài 5**
- Mức 1: thay `best = inf` bằng `best = None`, thay `1 + dfs(...)` bằng `[tx] + dfs(...)`.
- Mức 2: nếu `debts[start] > 0` thì `tx = [people[i], people[start], debts[start]]` (người `i` trả); ngược lại `tx = [people[start], people[i], -debts[start]]`.
- Mức 3: so sánh `len(cand) < len(best)`; lấy `amount = debts[start]` **trước** khi sửa `debts[i]`.

**Bài 6**
- Mức 1: `ok = [False] * (1 << n)`, `ok[0] = True`; duyệt `mask` tăng dần, chỉ "lan" từ những `mask` có `ok[mask]`.
- Mức 2: `filled = total[mask] % target` là lượng đã đổ vào giỏ hiện tại; thêm `nums[i]` chỉ khi `filled + nums[i] <= target`.
- Mức 3: nếu `ok[full]` thì mọi giỏ đều vừa khít (vì tổng `= k · target` và không giỏ nào tràn).

## 18. Đáp án tham khảo

### Đáp án Bài 1

1. `{0: 0, 1: 0, 2: 0}` — ai cũng đưa 5 nhận 5 → **0** giao dịch.
2. **1** (một người nợ, một người được nợ).
3. P0: `+3 + 3 − 6 = 0`, P1: `−3`, P2: `−3`, P3: `+6` → `nonzero_debts = [-3, -3, 6]`. Không tách được nhóm tổng 0 nhỏ hơn → `G = 1`, đáp án `3 − 1 = 2`: P3 trả P1 3 đô và trả P2 3 đô.
4. `[3, −3, 4, −2, −2]`: tổng tiền tố `3, 0, 4, 2, 0` → 2 số 0 → `5 − 2 = 3`. `[4, −2, 3, −2, −3]`: `4, 2, 5, 3, 0` → 1 số 0 → `4`.
5. `0b00110` → bit 1 và bit 2 → tập `{P1, P2}` = `{3, −3}`, `total = 0`.

### Đáp án Bài 2

a) `{6, −2, −4}`, `{5, −5}`, `{1, −1}` → `G = 3`, đáp án `7 − 3 = 4`.

b) Tổng tiền tố `6, 4, 0, 5, 0, 1, 0` có 3 số 0 → `7 − 3 = 4` (thứ tự gốc tình cờ đã tối ưu!).

c) Tham lam: `P4 trả P0 5` → `P2 trả P3 4` → `P1 trả P0 1` → `P1 trả P3 1` → `P6 trả P5 1`: **5** giao dịch (không tối ưu). Cell dưới kiểm chứng.

In [32]:
# Bài 1
assert net_balances([[0, 1, 5], [1, 2, 5], [2, 0, 5]]) == {0: 0, 1: 0, 2: 0}
assert Solution().minTransfers([[0, 1, 5], [1, 2, 5], [2, 0, 5]]) == 0
assert Solution().minTransfers([[0, 1, 7]]) == 1
assert nonzero_debts([[0, 1, 3], [0, 2, 3], [3, 0, 6]]) == [-3, -3, 6]
assert Solution().minTransfers([[0, 1, 3], [0, 2, 3], [3, 0, 6]]) == 2
assert chain_cost([3, -3, 4, -2, -2]) == 3 and chain_cost([4, -2, 3, -2, -3]) == 4
dd = [4, 3, -3, -2, -2]
assert [i for i in range(5) if 0b00110 >> i & 1] == [1, 2] and dd[1] + dd[2] == 0

# Bài 2
d2 = [6, -2, -4, 5, -5, 1, -1]
bal2 = {p: b for p, b in enumerate(d2)}
assert max_zero_groups_bruteforce(d2) == 3
assert min_transfers_backtrack(d2) == min_transfers_bitmask(d2) == 4
assert chain_cost(d2) == 4 and len(settle_chain(bal2)) == 4
g2 = settle_greedy(bal2, verbose=True)
print("tham lam:", len(g2), "giao dịch")
assert len(g2) == 5
print("Đáp án Bài 1, 2: PASS")

  P4 trả P0  5 -> còn lại {0: 1, 1: -2, 2: -4, 3: 5, 4: 0, 5: 1, 6: -1}
  P2 trả P3  4 -> còn lại {0: 1, 1: -2, 2: 0, 3: 1, 5: 1, 6: -1}
  P1 trả P0  1 -> còn lại {0: 0, 1: -1, 3: 1, 5: 1, 6: -1}
  P1 trả P3  1 -> còn lại {1: 0, 3: 0, 5: 1, 6: -1}
  P6 trả P5  1 -> còn lại {5: 0, 6: 0}
tham lam: 5 giao dịch
Đáp án Bài 1, 2: PASS


### Đáp án Bài 3 và Bài 4

Bài 3 là đúng code của `Solution` (mục 9) viết thành hàm; Bài 4 là `SolutionBitmask` (mục 10.7) viết thành hàm, ở đây dùng vòng `for` để tính `total` cho dễ đọc.

In [33]:
def min_transfers_exercise(transactions):
    bal = defaultdict(int)
    for f, t, a in transactions:
        bal[f] += a
        bal[t] -= a
    debts = [b for b in bal.values() if b != 0]
    n = len(debts)

    def dfs(start):
        while start < n and debts[start] == 0:
            start += 1
        if start == n:
            return 0
        best = float("inf")
        tried = set()
        for i in range(start + 1, n):
            if debts[i] * debts[start] < 0 and debts[i] not in tried:
                tried.add(debts[i])
                debts[i] += debts[start]
                best = min(best, 1 + dfs(start + 1))
                debts[i] -= debts[start]
                if debts[i] + debts[start] == 0:
                    break
        return best

    return dfs(0)


def min_transfers_bitmask_exercise(transactions):
    debts = nonzero_debts(transactions)
    n = len(debts)
    full = (1 << n) - 1
    total = [0] * (1 << n)
    dp = [0] * (1 << n)
    for mask in range(1, 1 << n):
        for i in range(n):
            if mask >> i & 1:
                prev = mask ^ (1 << i)
                total[mask] = total[prev] + debts[i]
                dp[mask] = max(dp[mask], dp[prev])
        dp[mask] += total[mask] == 0
    return n - dp[full]


for f in (min_transfers_exercise, min_transfers_bitmask_exercise):
    assert f(ex1) == 2
    assert f(ex2) == 1
    assert f(tricky) == 3
    assert f([[0, 1, 5], [1, 2, 5], [2, 0, 5]]) == 0
    assert f([[0, 1, 3], [2, 3, 3]]) == 2
rng = random.Random(7)
for _ in range(300):
    tr = random_transactions(rng, rng.choice([4, 6, 12]), rng.choice([3, 5, 100]))
    assert min_transfers_exercise(tr) == min_transfers_bitmask_exercise(tr) == Solution().minTransfers(tr)
print("Đáp án Bài 3, 4: PASS (kể cả 300 test ngẫu nhiên)")

Đáp án Bài 3, 4: PASS (kể cả 300 test ngẫu nhiên)


### Đáp án Bài 5 — Kế hoạch thanh toán tối ưu bằng quay lui

`dfs` giờ trả về **list giao dịch** thay vì số. Lưu ý lấy `amount = debts[start]` trước khi sửa `debts[i]`, và chiều giao dịch phụ thuộc dấu của `amount` (bảng ở mục 7.1).

In [34]:
def optimal_plan(transactions):
    bal = net_balances(transactions)
    people = [p for p, b in bal.items() if b != 0]
    debts = [bal[p] for p in people]
    n = len(debts)

    def dfs(start):
        while start < n and debts[start] == 0:
            start += 1
        if start == n:
            return []
        best = None
        tried = set()
        for i in range(start + 1, n):
            if debts[i] * debts[start] < 0 and debts[i] not in tried:
                tried.add(debts[i])
                amount = debts[start]
                if amount > 0:
                    tx = [people[i], people[start], amount]      # i trả start
                else:
                    tx = [people[start], people[i], -amount]     # start trả i
                debts[i] += amount
                cand = [tx] + dfs(start + 1)
                debts[i] -= amount
                if best is None or len(cand) < len(best):
                    best = cand
                if debts[i] + debts[start] == 0:
                    break
        return best

    return dfs(0)


for name, tr in [("ex1", ex1), ("ex2", ex2), ("tricky", tricky)]:
    plan = optimal_plan(tr)
    print(f"{name:7s}: {plan}")
    assert is_settled(tr, plan) and len(plan) == Solution().minTransfers(tr)
rng = random.Random(11)
for _ in range(300):
    tr = random_transactions(rng, rng.choice([4, 6, 12]), rng.choice([3, 5, 100]))
    plan = optimal_plan(tr)
    assert is_settled(tr, plan) and len(plan) == Solution().minTransfers(tr), tr
print("Đáp án Bài 5: PASS")

ex1    : [[1, 0, 5], [1, 2, 5]]
ex2    : [[1, 0, 4]]
tricky : [[3, 0, 4], [2, 1, 3], [4, 3, 2]]


Đáp án Bài 5: PASS


### Đáp án Bài 6 — Partition to K Equal Sum Subsets (DP bitmask)

`ok[mask]` = có thể xếp **tập** `mask` vào các giỏ theo luật "đổ đầy giỏ hiện tại rồi mới sang giỏ mới, không bao giờ tràn". Lượng đã đổ trong giỏ hiện tại là `total[mask] % target`. Giống hệt Cách 4: thứ tự bên trong tập không quan trọng, chỉ tập mới quan trọng. Độ phức tạp `O(2^n · n)` với `n <= 16`.

In [35]:
def can_partition_k_subsets(nums, k):
    s = sum(nums)
    if s % k != 0:
        return False
    target = s // k
    if max(nums) > target:
        return False
    n = len(nums)
    total = [0] * (1 << n)
    ok = [False] * (1 << n)
    ok[0] = True
    for mask in range(1 << n):
        if mask:
            low = (mask & -mask).bit_length() - 1
            total[mask] = total[mask ^ (1 << low)] + nums[low]
        if not ok[mask]:
            continue
        filled = total[mask] % target
        for i in range(n):
            if not mask >> i & 1 and filled + nums[i] <= target:
                ok[mask | (1 << i)] = True
    return ok[(1 << n) - 1]


assert can_partition_k_subsets([4, 3, 2, 3, 5, 2, 1], 4) is True
assert can_partition_k_subsets([1, 2, 3, 4], 3) is False
assert can_partition_k_subsets([2, 2, 2, 2, 3, 4, 5], 4) is False
assert can_partition_k_subsets([5], 1) is True
assert can_partition_k_subsets([1, 1, 1, 1, 2, 2, 2, 2], 4) is True
print("Đáp án Bài 6: PASS")

Đáp án Bài 6: PASS


## 19. Template cuối cùng nên nhớ

```python
class Solution:
    def minTransfers(self, transactions: List[List[int]]) -> int:
        bal = defaultdict(int)
        for f, t, a in transactions:          # 1) số dư ròng
            bal[f] += a
            bal[t] -= a
        debts = [b for b in bal.values() if b != 0]   # 2) bỏ người số dư 0
        n = len(debts)

        def dfs(start):
            while start < n and debts[start] == 0:    # 3) bỏ qua người đã xong
                start += 1
            if start == n:
                return 0
            best, tried = float("inf"), set()          #    tried MỚI cho mỗi tầng
            for i in range(start + 1, n):
                if debts[i] * debts[start] < 0 and debts[i] not in tried:   # trái dấu, chưa thử
                    tried.add(debts[i])
                    debts[i] += debts[start]           # CHỌN
                    best = min(best, 1 + dfs(start + 1))   # ĐỆ QUY (start + 1, không phải i + 1)
                    debts[i] -= debts[start]           # HOÀN TÁC
                    if debts[i] + debts[start] == 0:   # triệt tiêu đúng -> tối ưu
                        break
            return best

        return dfs(0)
```

Bản DP bitmask: *`total[mask]` = tổng tập; `dp[mask] = max(dp[mask ^ (1<<i)]) + (total[mask] == 0)`; trả `n − dp[full]`.*

**Câu thần chú**: *"Gộp thành số dư, bỏ số 0. Đáp án = số người − số nhóm tổng 0 nhiều nhất. Người đầu tiên còn lệch chuyển hết cho một người trái dấu — chọn, đệ quy, hoàn tác; bỏ trùng, gặp cặp triệt tiêu thì dừng."*

## 20. Checklist tự kiểm tra trước khi submit

- Quy ước dấu nhất quán: `bal[f] += a`, `bal[t] -= a` (đổi ngược cả hai cũng được, miễn nhất quán)?
- Đã **bỏ người số dư 0** khỏi `debts`, và trong `dfs` có `while` bỏ qua người đã về 0 trước base case?
- Chỉ ghép với người **trái dấu** (`debts[i] * debts[start] < 0`)?
- Mỗi `debts[i] += debts[start]` có đúng một `debts[i] -= debts[start]` ngay sau lời gọi đệ quy?
- Gọi đệ quy `dfs(start + 1)`, **không** phải `dfs(i + 1)`?
- `tried = set()` tạo **bên trong** `dfs` (mỗi tầng một set), và `break` chỉ khi `debts[i] + debts[start] == 0`, đặt **sau** khi hoàn tác?
- Nếu dùng bitmask: `+1` **chỉ khi** `total[mask] == 0`, và trả `n − dp[full]` (không phải `dp[full]`)?
- Đã test: 2 ví dụ của đề, vòng tròn (đáp án 0), một giao dịch, hai cặp độc lập, giá trị trùng, phản ví dụ tham lam `[[0,2,4],[1,3,2],[1,4,1],[2,4,1]]` (đáp án 3), và một bộ 8 giao dịch ID tới 11?